# Where to measure? Process-informed sparse metrology for plasma-etch uniformity (v2.1)

**Data:** *A Multi-Model Dataset for BOSCH Plasma-Etching* (Sayyed et al., Zenodo [17122442](https://zenodo.org/records/17122442), CC BY 4.0): 96 wafers in 10 lots on an SPTS Rapier DRIE tool, 5 Hz machine signals, optical emission spectra (OES), and Si etch depth measured at 89 sites (offline, Feb 2025) and at 9 sites (in-line, on the process day).

**Question.** A fab measures a few sites per wafer. Which sites should they be, how well can the full 89-site map and its uniformity be reconstructed from them on **lots the model has never seen**, and how much do process information (wafer order, conditioning recipe, machine signals, OES) add?

**What this notebook produces for the paper**

| RQ | Question | Output |
|---|---|---|
| audit | Are the two measurement campaigns consistent? How do the maps vary? | Fig. 1-3, `results.json` → `audit` |
| RQ1 | Where to measure without process data (standard 1/5/9-site layouts, the same layouts with edge sites, FSCA, QR, greedy Bayesian, random)? Is the learned covariance needed, or is the mean map enough (fair baselines)? | Tables `rq1`, `rq1_estimators`, Fig. 4a, Fig. 5 |
| RQ2 | What is process information worth, in measured sites? | Table `rq2`, Fig. 4b |
| RQ3 | Can process information choose the sites per wafer? | Table `rq3` |
| RQ4 | Are the uncertainties trustworthy on unseen lots (Gaussian vs conformal)? | Table `rq4`, Fig. 6 |
| RQ5 | Can process information decide which wafers need no measurement? | Table `rq5` |
| RQ6 | Do the real in-line 9-site measurements reconstruct the offline 89-site map? | Table `rq6` |
| check | Do the conclusions hold when every lot is predicted from earlier lots only? | Table `forward_in_time` |

**Protocol.** Leave-one-lot-out everywhere: models, covariances, site designs, conformal calibration and penalties are fitted on the other nine lots only; scores are averaged per lot and then over lots, with 95 % lot-bootstrap intervals.

**How to run.** Put all files of the Zenodo record in one folder together with this notebook (or in a Google Drive folder and set `USE_GOOGLE_DRIVE = True`), then *Runtime → Run all*. A CPU runtime is enough. The first run reads all `Day_*.nc` OES files once (10-20 min) and caches the features; the experiments take another 15-25 min. At the end, send back the printed `results.json` (and the zip of `results_sparse_metrology/`, which now also holds a small `oes_debug.npz` for checking the spectra).

**Changes in v2** (after the first run on the full data)
* **OES fixed.** v1 judged every 40 ms spectrum on its own and discarded almost all of them (single spectra are too noisy). v2 builds every feature from averages: each wafer's spectra are folded at its Bosch cycle period (taken from the machine signals) and the SF6 / C4F8 phase window is the one that best separates the spectra. New cache files, so nothing from v1 is reused.
* **Stronger virtual metrology.** The wafer offset (≈90 % of the wafer-to-wafer variation) is predicted by an elastic net instead of ridge regression (`OFFSET_MODEL`); ridge is kept as a sensitivity setting.
* **Fair baselines.** Besides IDW (no history), the learned-covariance model is compared with baselines that also use the training lots: mean map + offset, mean map + IDW of the residuals, stationary (exponential) kriging.
* **Edge-inclusive layouts.** The standard 5/9-site layouts with their outer sites moved to the edge ring (r = 95 mm).
* **Forward-in-time check, corrected campaign-audit numbers, failure rate of skipped wafers in RQ5.**

## 0. Settings and setup

In [ ]:
# ---------------------------------------------------------------- settings (edit here)
DATA_DIR = "."                 # folder that holds the files of Zenodo record 17122442 (same folder as this notebook)
USE_GOOGLE_DRIVE = False       # True: mount Google Drive and read the data from DRIVE_DATA_DIR instead
DRIVE_DATA_DIR = "/content/drive/MyDrive/bosch_etch"
OUT_DIR = None                 # default: <DATA_DIR>/results_sparse_metrology

TARGET = "si_etch"             # map to reconstruct: Si etch depth (um); "stepheight" is run as a sensitivity check
K_MAX = 12                     # largest number of measured sites studied
NOISE_SD = 0.03                # um, measurement noise of one site (the campaign comparison gives ~0.03 per tool)
N_MODES = 4                    # shape modes predicted by the virtual-metrology prior
OFFSET_MODEL = "enet"          # wafer-offset model of the prior: "enet" (elastic net) or "ridge" (v1 of this notebook)
N_RANDOM = 20                  # random layouts averaged per k
ALPHA = 0.10                   # 90 % prediction intervals
TAUS = [0.10, 0.15, 0.20, 0.30]   # um, tolerances for the measure-or-skip policy
USE_OES = True                 # use the Day_*.nc spectra if they are in DATA_DIR (first run: ~10-30 min, then cached)
OES_MIN_COVERAGE = 0.8         # use OES only if at least this share of the wafers has spectra
OUTLIER_Z = 10.0               # flag gross measurement errors (> 10 robust s.d.); None keeps the raw maps
RUN_SENSITIVITY = True
N_BOOT = 2000
SEED = 0

# ---------------------------------------------------------------- setup
import importlib.util, os, subprocess, sys, time
for _pkg, _pip in (("netCDF4", "netCDF4"), ("sklearn", "scikit-learn"), ("openpyxl", "openpyxl")):
    if importlib.util.find_spec(_pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pip], check=True)
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = DRIVE_DATA_DIR
OUT_DIR = OUT_DIR or os.path.join(DATA_DIR, "results_sparse_metrology")
for sub in ("", "raw", "tables", "figures"):
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)
NOISE_VAR = NUGGET = NOISE_SD ** 2
print("data folder:", os.path.abspath(DATA_DIR))
for f in sorted(os.listdir(DATA_DIR)):
    p = os.path.join(DATA_DIR, f)
    if os.path.isfile(p):
        print(f"  {f:40s} {os.path.getsize(p) / 1e6:9.1f} MB")
print("results go to:", os.path.abspath(OUT_DIR))

## Core code
Data loading and audit, features, the virtual-metrology prior and Gaussian conditioning, the site-design methods, the experiments and the figures. Run these cells; nothing is computed yet.

In [ ]:
# ===================================================================== core 1/3: files, measurements, audit
import glob
import json
import os
import re
import time
import warnings

import numpy as np
import pandas as pd

FILES = {
    "m89": "Si_Oxide_etch_89_points.csv",
    "m9": "Si_Oxide_etch_9_points.csv",
    "lots": "Lot_status.xlsx",
    "proc": "Process_data.nc",
    "proc_dict": "Dictionary_process.nc",
    "oes_dict": "Dictionary_OES.nc",
}
COLS = "ABCDEFGHIJK"            # x = -95 ... +95 mm on the 19 mm grid; column F is the centre
STANDARD_LAYOUTS = {             # the dataset's 9-point layout (Wafer_layout.pdf) and its usual 1- and 5-point subsets
    1: ["F6"],
    5: ["F6", "F2", "F10", "B6", "J6"],
    9: ["F2", "D4", "H4", "B6", "F6", "J6", "D8", "H8", "F10"],
}
EDGE_LAYOUTS = {                 # same patterns with the outer sites moved to the edge ring (r = 95 mm instead of 76 mm)
    5: ["F6", "F1", "F11", "A6", "K6"],
    9: ["F1", "D4", "H4", "A6", "F6", "K6", "D8", "H8", "F11"],
}
D4 = {                           # the eight symmetries of the square measurement grid
    "identity": lambda x, y: (x, y), "rot90": lambda x, y: (-y, x), "rot180": lambda x, y: (-x, -y),
    "rot270": lambda x, y: (y, -x), "flip_x": lambda x, y: (-x, y), "flip_y": lambda x, y: (x, -y),
    "transpose": lambda x, y: (y, x), "antitranspose": lambda x, y: (-y, -x),
}
R_INNER, R_EDGE, R_CENTER = 86.0, 90.0, 30.0   # mm: inner sites (69), edge ring (20), centre disk


def find_file(data_dir, name, required=True):
    """Find a dataset file in data_dir (also accepts names with a prefix, e.g. '1a2b3c-Process_data.nc')."""
    exact = os.path.join(data_dir, name)
    if os.path.exists(exact):
        return exact
    hits = sorted(glob.glob(os.path.join(data_dir, "*" + name)))
    if hits:
        return hits[0]
    if required:
        raise FileNotFoundError(f"{name} not found in {data_dir}")
    return None


def site_label(x_mm, y_mm):
    """Grid label as in Wafer_layout.pdf: column A..K from x, row 1..11 from y (row 1 = top, y = +95 mm)."""
    return f"{COLS[int(round(x_mm / 19.0)) + 5]}{int(round((95.0 - y_mm) / 19.0)) + 1}"


def load_lot_status(data_dir):
    """Lot table (date, wafers, measured sets, conditioning recipe) and the FRT NaN table of Lot_status.xlsx."""
    raw = pd.read_excel(find_file(data_dir, FILES["lots"]), header=None)
    lots, start = [], None
    for i, r in raw.iterrows():
        m = re.match(r"Lot\s*(\d+)", str(r[0]).strip())
        if m:
            typ = str(r[4]).strip()
            mc = re.match(r"(\d+)", typ)
            lots.append({"lot": int(m.group(1)), "date_text": str(r[1]).strip(), "wafers_processed": r[2],
                         "measured": " ".join(str(r[3]).split()), "type": typ,
                         "n_conditioning": int(mc.group(1)) if mc else np.nan,
                         "cond_substrate": "SiO2" if "SiO2" in typ else ("Si" if "Si" in typ else "chuck")})
        if str(r[0]).strip().lower() == "lot" and str(r[1]).strip().lower() == "wafer":
            start = i + 1
    nan_rows = []
    if start is not None:
        for _, r in raw.iloc[start:].iterrows():
            if pd.notna(r[0]) and pd.notna(r[1]) and pd.notna(r[2]):
                nan_rows.append({"lot": int(r[0]), "wafer": int(r[1]), "frt_nan_count": int(r[2])})
    return pd.DataFrame(lots).set_index("lot"), pd.DataFrame(nan_rows)


def load_measurements(data_dir, target="si_etch"):
    """89-point maps (wafers x sites, um), the in-line 9-point campaign, the site table and per-wafer metadata."""
    d89 = pd.read_csv(find_file(data_dir, FILES["m89"]))
    p9 = find_file(data_dir, FILES["m9"], required=False)
    d9 = pd.read_csv(p9) if p9 else None
    maps = d89.pivot(index="experiment_key", columns=["X", "Y"], values=target).sort_index()
    if maps.isna().any().any():
        raise ValueError("missing values in the 89-point maps")
    cols_xy = maps.columns
    xy = np.array(list(cols_xy), dtype=float) / 1000.0                       # um -> mm
    sites = pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1]})
    sites["r"] = np.hypot(sites.x, sites.y)
    sites["label"] = [site_label(a, b) for a, b in zip(sites.x, sites.y)]
    maps.columns = sites.label.values
    meta = d89.groupby("experiment_key")[["lot_number", "wafer_number"]].first().loc[maps.index]
    meta = meta.rename(columns={"lot_number": "lot", "wafer_number": "wafer"})
    meta["date"] = [k.split("_")[0] for k in meta.index]
    lots, nan_tab = load_lot_status(data_dir)
    meta = meta.join(lots[["n_conditioning", "cond_substrate", "type"]], on="lot")
    extra = {}
    for col in ["stepheight", "postox_thickness", "postox_thickness_nan", "preox_thickness", "oxide_etch", "si_etch"]:
        if col in d89.columns:
            m = d89.pivot(index="experiment_key", columns=["X", "Y"], values=col).sort_index().reindex(columns=cols_xy)
            m.columns = sites.label.values
            extra[col] = m.loc[maps.index]
    return {"maps": maps, "sites": sites, "meta": meta, "d89": d89, "d9": d9, "lots": lots,
            "frt_nan": nan_tab, "extra": extra, "target": target}


def campaign_pairs(d9, d89, value, transform="identity", keys=None):
    """Pair every in-line 9-point value with the offline 89-point value at the transformed location."""
    f = D4[transform]
    m89 = d89.set_index(["experiment_key", "X", "Y"])[value]
    rows = []
    for _, r in d9.iterrows():
        if keys is not None and r.experiment_key not in keys:
            continue
        x2, y2 = f(r.X, r.Y)
        k = (r.experiment_key, int(x2), int(y2))
        if k in m89.index:
            rows.append((r.experiment_key, r.loc_id, int(x2), int(y2), r[value], m89.loc[k]))
    return pd.DataFrame(rows, columns=["wafer", "loc_id", "X89", "Y89", "v9", "v89"])


def audit_campaigns(d9, d89, value="si_etch", keys=None):
    """Agreement of the two measurement campaigns under each of the 8 grid symmetries (best first)."""
    out = []
    for name in D4:
        j = campaign_pairs(d9, d89, value, name, keys)
        if len(j) == 0:
            continue
        j["d"] = j.v9 - j.v89
        j["d_site"] = j.d - j.groupby("loc_id").d.transform("mean")           # remove a per-site offset
        j["d_wafer"] = j.d - j.groupby("wafer").d.transform("mean")           # remove a per-wafer offset
        pc = j.groupby("wafer").apply(
            lambda g: np.corrcoef(g.v9, g.v89)[0, 1] if g.v9.std() > 0 and g.v89.std() > 0 else np.nan,
            include_groups=False)
        out.append({"transform": name, "pairs": len(j), "wafers": j.wafer.nunique(), "mean_diff_um": j.d.mean(),
                    "resid_sd_after_wafer_offset_um": j.d_wafer.std(),
                    "resid_sd_after_site_offset_um": j.d_site.std(),
                    "median_within_wafer_corr": pc.median()})
    return pd.DataFrame(out).sort_values("resid_sd_after_wafer_offset_um").reset_index(drop=True)


def uniformity_metrics(Ymat, r):
    """Per-wafer uniformity metrics of maps (rows) on all sites and on the inner (edge-excluded) sites."""
    Ymat = np.atleast_2d(Ymat)
    inner, edge, centre = r <= R_INNER, r >= R_EDGE, r <= R_CENTER
    mean = Ymat.mean(1)
    sd = Ymat.std(1)
    return pd.DataFrame({
        "mean": mean, "sd": sd, "cv_pct": 100 * sd / mean, "range": Ymat.max(1) - Ymat.min(1),
        "edge_minus_centre": Ymat[:, edge].mean(1) - Ymat[:, centre].mean(1),
        "sd_inner": Ymat[:, inner].std(1), "cv_inner_pct": 100 * Ymat[:, inner].std(1) / Ymat[:, inner].mean(1),
    })


def drift_table(maps, meta):
    """Within-lot drift: correlation of the wafer-mean etch depth with the processing order, per lot."""
    df = pd.DataFrame({"lot": meta.lot.values, "order": meta.wafer.values, "mean": maps.values.mean(1)})
    rows = []
    for g, d in df.groupby("lot"):
        slope = np.polyfit(d.order, d["mean"], 1)[0] if len(d) > 2 else np.nan
        rows.append({"lot": g, "wafers": len(d), "corr_mean_vs_order": d[["order", "mean"]].corr().iloc[0, 1],
                     "slope_um_per_wafer": slope, "lot_mean_um": d["mean"].mean()})
    return pd.DataFrame(rows).set_index("lot")

In [ ]:
# ===================================================================== core 2/3: features (context, process, OES)
def _decoder(data_dir, key):
    """Dictionary array of the 16-bit encoding (Readme, sections 2.1-2.2), or None if the file is missing."""
    p = find_file(data_dir, FILES[key], required=False)
    if p is None:
        return None
    from netCDF4 import Dataset
    with Dataset(p, "r") as ds:
        return np.asarray(ds["data"][:])


def _codes(var, n_dict=None):
    """Read an encoded NetCDF variable as integer codes; rows with masked (missing) entries are dropped."""
    arr = var[:]
    keep = None
    if np.ma.isMaskedArray(arr):
        m = np.ma.getmaskarray(arr)
        if m.any():
            keep = ~m.reshape(m.shape[0], -1).any(1)
        arr = np.ma.getdata(arr)
    arr = np.asarray(arr)
    if keep is not None:
        arr = arr[keep]
    if n_dict is not None:
        arr = np.clip(arr, 0, n_dict - 1)
    return arr, keep


def _segments(on):
    """[start, end) index pairs of the runs of True in a boolean array."""
    on = np.asarray(on, dtype=bool)
    if not on.any():
        return np.zeros((0, 2), dtype=int)
    d = np.diff(np.r_[0, on.astype(int), 0])
    return np.c_[np.flatnonzero(d == 1), np.flatnonzero(d == -1)]


def bosch_phases(df, gas_cols):
    """Label every 5 Hz sample as SF6 phase, C4F8 phase or neither.

    The two gases that switch on and off ~100 times are found automatically; the one with the longer pulses
    (4.5 s vs 1.5 s) is SF6. Works on decoded values and on raw (monotone) codes alike.
    """
    cand = []
    for c in gas_cols:
        v = df[c].to_numpy(dtype=float)
        lo, hi = np.percentile(v, 2), np.percentile(v, 98)
        rng = hi - lo
        if rng <= 0:
            continue
        near = ((np.abs(v - lo) < 0.15 * rng) | (np.abs(v - hi) < 0.15 * rng)).mean()
        on = v > lo + 0.5 * rng
        seg = _segments(on)
        if near > 0.6 and 40 <= len(seg) <= 250:
            cand.append((c, on, len(seg), near, np.median(seg[:, 1] - seg[:, 0])))
    if len(cand) < 2:
        return None, {"ok": False}
    cand.sort(key=lambda t: -t[3])
    a, b = cand[0], cand[1]
    sf6, c4 = (a, b) if a[4] >= b[4] else (b, a)
    lab = np.full(len(df), "", dtype=object)
    lab[sf6[1] & ~c4[1]] = "sf6"
    lab[c4[1] & ~sf6[1]] = "c4f8"
    return lab, {"ok": True, "sf6_channel": sf6[0], "c4f8_channel": c4[0], "n_sf6_pulses": sf6[2],
                 "n_c4f8_pulses": c4[2]}


def _group_key(name):
    """'Day_2024_07_02_Wafer_01' -> '2024-07-02_01' (the experiment_key of the measurement files)."""
    m = re.match(r"Day_(\d{4})_(\d{2})_(\d{2})_Wafer_(\d+)", name)
    return f"{m.group(1)}-{m.group(2)}-{m.group(3)}_{int(m.group(4)):02d}" if m else None


def process_features(data_dir, keys=None, verbose=True):
    """Per-wafer summaries of the 5 Hz machine signals (Process_data.nc + Dictionary_process.nc).

    For every channel recorded on all wafers and in each Bosch phase: median, IQR, the slope of the per-pulse
    median over the ~100 cycles and the last-10 minus first-10 pulses; plus cycle-timing features.
    """
    from netCDF4 import Dataset
    dec = _decoder(data_dir, "proc_dict")
    if dec is None:
        warnings.warn("Dictionary_process.nc not found - falling back to the raw 16-bit codes, which are only "
                      "monotone in the physical values. Put the dictionary next to the notebook for real results.")
    rows, infos = {}, {}
    with Dataset(find_file(data_dir, FILES["proc"]), "r") as ds:
        groups = list(ds.groups)
        common = sorted(set.intersection(*[set(str(f) for f in ds.groups[g]["feature"][:]) for g in groups]))
        for g in groups:
            key = _group_key(g)
            if key is None or (keys is not None and key not in keys):
                continue
            grp = ds.groups[g]
            names = [str(f) for f in grp["feature"][:]]
            enc, keep = _codes(grp["data"], None if dec is None else len(dec))
            val = dec[enc] if dec is not None else enc.astype(float)
            df = pd.DataFrame(val, columns=names)[common].astype(float)
            t = np.asarray(grp["times"][:], dtype=float)
            if keep is not None:
                t = t[keep]
            lab, info = bosch_phases(df, [c for c in common if re.search(r"Gas\d+Flow$", c)])
            infos[key] = info
            f = {}
            if lab is not None:
                seg = {"sf6": _segments(lab == "sf6"), "c4f8": _segments(lab == "c4f8")}
                f["n_cycles"] = len(seg["sf6"])
                for ph in ("sf6", "c4f8"):
                    s = seg[ph]
                    f[f"{ph}_pulse_s"] = float(np.median(t[np.minimum(s[:, 1], len(t) - 1)] - t[s[:, 0]]))
                f["etch_window_s"] = float(t[seg["sf6"][-1, 1] - 1] - t[seg["sf6"][0, 0]])
                info.update({"n_cycles": f["n_cycles"], "sf6_pulse_s": f["sf6_pulse_s"], "c4f8_pulse_s": f["c4f8_pulse_s"],
                             "etch_window_s": f["etch_window_s"]})
                t_ep = _epoch_s(grp["times"])                       # cycle timing, used to fold the OES of this wafer
                if keep is not None:
                    t_ep = t_ep[keep]
                st = t_ep[seg["sf6"][:, 0]]
                en = t_ep[np.minimum(seg["sf6"][:, 1], len(t_ep) - 1)]
                if len(st) >= 12:
                    ii = np.arange(2, len(st) - 2)
                    per = float(np.polyfit(ii, st[2:-2], 1)[0])
                    info.update({"cycle_period_s": per, "sf6_duty": float(np.mean(en[2:-2] - st[2:-2]) / per),
                                 "t0_sf6_epoch_s": float(st[2])})
                for c in common:
                    v = df[c].to_numpy()
                    for ph in ("sf6", "c4f8"):
                        x = v[lab == ph]
                        if len(x) == 0:
                            continue
                        f[f"{c}|{ph}|med"] = np.median(x)
                        f[f"{c}|{ph}|iqr"] = np.subtract(*np.percentile(x, [75, 25]))
                        cyc = np.array([np.median(v[a:b]) for a, b in seg[ph] if b > a])
                        if len(cyc) >= 20:
                            f[f"{c}|{ph}|slope"] = np.polyfit(np.arange(len(cyc)), cyc, 1)[0]
                            f[f"{c}|{ph}|last-first"] = np.median(cyc[-10:]) - np.median(cyc[:10])
            rows[key] = f
    F = pd.DataFrame.from_dict(rows, orient="index").sort_index()
    F = F.loc[:, F.notna().all()]
    F = F.loc[:, F.std() > 1e-12 * (F.abs().max() + 1.0)]
    F.columns = [c.replace("Stat3_Etch_MV_", "") for c in F.columns]
    if verbose:
        ok = sum(1 for v in infos.values() if v.get("ok"))
        first = next(iter(infos.values()))
        print(f"process features: {F.shape[1]} for {F.shape[0]} wafers; Bosch phases found on {ok}/{len(infos)} wafers "
              f"(SF6 = {first.get('sf6_channel')}, C4F8 = {first.get('c4f8_channel')})")
    return F, infos


OES_LINES_NM = {                 # lines/bands commonly monitored in SF6/C4F8 plasmas (window +/- 0.8 nm)
    "F_685.6": 685.6, "F_690.2": 690.2, "F_703.7": 703.7, "F_712.8": 712.8, "F_739.9": 739.9,
    "O_777.4": 777.4, "O_844.6": 844.6, "SiF_436.8": 436.8, "SiF_440.1": 440.1, "C2_516.5": 516.5,
    "C2_563.6": 563.6, "CS_257.6": 257.6, "CF2_251.9": 251.9, "CF2_262.8": 262.8, "CF2_271.2": 271.2,
    "S2_365": 365.0, "H_656.3": 656.3, "N2_337.1": 337.1, "CO_483.5": 483.5, "Ar_750.4": 750.4,
}
TREND_LINES = ("F_703.7", "SiF_440.1", "C2_516.5", "O_777.4")


def _epoch_s(tvar):
    """NetCDF time variable -> float seconds since 1970-01-01 (naive times taken as UTC); raw values if that fails."""
    from netCDF4 import num2date
    raw = np.asarray(tvar[:], dtype=float)
    try:
        d = num2date(raw, tvar.units, getattr(tvar, "calendar", "standard"),
                     only_use_cftime_datetimes=False, only_use_python_datetimes=True)
        return pd.to_datetime(np.asarray(d)).values.astype("datetime64[ns]").astype(np.int64) / 1e9
    except Exception:
        return raw


def _decode_raw(var, dec):
    """Decode a dictionary-encoded variable exactly like the Readme (decoder[encoded]): no masking, 16-bit codes
    stored as signed integers are read as unsigned. Returns (values float32, info)."""
    var.set_auto_maskandscale(False)
    arr = np.asarray(var[:])
    info = {"diag|code_dtype": str(arr.dtype), "diag|code_min": int(arr.min()), "diag|code_max": int(arr.max())}
    if np.issubdtype(arr.dtype, np.signedinteger) and arr.min() < 0:
        arr = arr.astype(np.int64) & 0xFFFF
    if dec is None:
        return arr.astype(np.float32), info
    over = int((arr >= len(dec)).sum())
    if over:
        info["diag|codes_beyond_dictionary"] = over
        arr = np.minimum(arr, len(dec) - 1)
    return dec[arr].astype(np.float32, copy=False), info


def _fold_score(phase, z, nbin=48):
    """Variance of the phase-folded means of z (epoch folding)."""
    b = np.minimum((phase * nbin).astype(int), nbin - 1)
    cnt = np.bincount(b, minlength=nbin)
    s = np.bincount(b, weights=z, minlength=nbin)
    ok = cnt > 0
    mu = s[ok] / cnt[ok]
    return float((cnt[ok] * (mu - z.mean()) ** 2).sum() / cnt.sum())


def _period_from_oes(t, z, lo=4.0, hi=9.0):
    """Bosch cycle period from the OES itself: epoch folding of the leading spectral component (coarse, then fine)."""
    tt = t - t[0]
    grid = np.arange(lo, hi, 0.005)
    sc = np.array([_fold_score((tt / p) % 1.0, z) for p in grid])
    p0 = grid[np.argmax(sc)]
    fine = np.arange(p0 - 0.01, p0 + 0.01, 0.0002)
    sc2 = np.array([_fold_score((tt / p) % 1.0, z) for p in fine])
    return float(fine[np.argmax(sc2)]), float(sc.max() / np.median(sc))


def _phase_labels(t, Bz, period, duty0, t_ref, nbin=96, duty_span=0.06):
    """Assign SF6 / C4F8 to every sample by folding at the Bosch period and finding the window (start, duty) that
    best separates the spectra into two groups (between-group sum of squares of the standardised binned spectra).
    Returns (is_sf6, phase, info)."""
    ph = ((t - t_ref) / period) % 1.0
    b = np.minimum((ph * nbin).astype(int), nbin - 1)
    cnt = np.bincount(b, minlength=nbin).astype(float)
    S = np.stack([np.bincount(b, weights=Bz[:, j], minlength=nbin) for j in range(Bz.shape[1])], 1)
    tot_s, tot_n = S.sum(0), cnt.sum()
    best = (-np.inf, 0, duty0)
    scores = []
    for d in np.arange(max(0.5, duty0 - duty_span), min(0.92, duty0 + duty_span) + 1e-9, 1.0 / nbin):
        w = int(round(d * nbin))
        for s0 in range(nbin):
            idx = (s0 + np.arange(w)) % nbin
            n1 = cnt[idx].sum()
            if n1 <= 0 or n1 >= tot_n:
                continue
            m1 = S[idx].sum(0) / n1
            m2 = (tot_s - S[idx].sum(0)) / (tot_n - n1)
            bss = n1 * (tot_n - n1) / tot_n * ((m1 - m2) ** 2).sum()
            scores.append(bss)
            if bss > best[0]:
                best = (bss, s0, w / nbin)
    _, s0, duty = best
    start = s0 / nbin
    is_sf6 = ((ph - start) % 1.0) < duty
    # distance (in s) of every sample from the nearest switching edge, to drop mixed spectra at the transitions
    x = (ph - start) % 1.0
    edge = np.minimum.reduce([x, np.abs(x - duty), 1.0 - x]) * period
    info = {"diag|phase_contrast": float(best[0] / (np.median(scores) + 1e-12)), "diag|sf6_fraction": float(is_sf6.mean()),
            "diag|duty": float(duty), "diag|phase_start": float(start)}
    return is_sf6, edge, info


def _oes_wafer_features(spec, wl, t, timing=None, lines=OES_LINES_NM, half_nm=0.8, bin_nm=10.0, edge_s=0.2):
    """Features of one wafer's OES record (samples x wavelengths, decoded) with sample times t (s).

    No per-spectrum thresholds: single 40 ms spectra are noisy, so everything is built from averages.
    * baseline of every spectrum = mean of the dimmest 10 % of pixels in the record-mean spectrum (no emission there);
    * plasma on = projection on the record-mean spectrum above 15 % of its median (records span the etch window);
    * SF6 / C4F8 phases by folding at the Bosch cycle period (from the process data of the same wafer if given,
      else from the spectra) and choosing the phase window that best separates the spectra;
    * features: emission share of 10 nm bands, continuum-subtracted line intensities, log total emission and early-to-
      late trends - over the whole etch and per phase.
    Returns (features, diagnostics, debug arrays)."""
    n, npx = spec.shape
    mean_spec = spec.mean(0, dtype=np.float64)
    dark = mean_spec <= np.percentile(mean_spec, 10)
    base = spec[:, dark].mean(1, dtype=np.float64)
    edges = np.arange(np.floor(wl.min()), wl.max() + bin_nm, bin_nm)
    bidx = np.clip(np.digitize(wl, edges) - 1, 0, len(edges) - 2)
    nb = len(edges) - 1
    Mb = np.zeros((npx, nb), dtype=np.float32)
    Mb[np.arange(npx), bidx] = 1.0
    npx_bin = Mb.sum(0)
    Mb /= np.maximum(npx_bin, 1)
    names = list(lines)
    Ml = np.zeros((npx, len(names)), dtype=np.float32)
    Ms = np.zeros((npx, len(names)), dtype=np.float32)
    for j, l0 in enumerate(lines.values()):
        w = np.abs(wl - l0) <= half_nm
        s = (np.abs(wl - l0) >= 1.5) & (np.abs(wl - l0) <= 3.0)
        if w.any() and s.any():
            Ml[w, j] = 1.0 / w.sum()
            Ms[s, j] = 1.0 / s.sum()
    B = spec @ Mb - base[:, None]                                     # band means above the baseline (n x nb)
    L = spec @ (Ml - Ms)                                               # continuum-subtracted line intensities (n x nl)
    tot = spec.sum(1, dtype=np.float64) - base * npx                   # total emission above the baseline
    templ = B.mean(0)
    amp = (B @ templ) / (templ @ templ + 1e-12)
    on = amp > 0.15 * np.median(amp)
    diag = {"diag|n_samples": int(n), "diag|on_fraction": float(on.mean()),
            "diag|dt_median_s": float(np.median(np.diff(t))) if n > 1 else np.nan,
            "diag|line_snr_F703": np.nan}
    debug = {"mean_spec": mean_spec.astype(np.float32)}
    if on.sum() < 200:
        diag["diag|phase_method"] = "too few plasma-on samples"
        return {}, diag, debug
    t, B, L, tot = t[on], B[on], L[on], tot[on]
    Bz = (B - B.mean(0)) / (B.std(0) + 1e-9)
    # Bosch phases
    method, info = "none", {}
    if timing is not None and np.isfinite(timing.get("period", np.nan)):
        period, duty0, t_ref = timing["period"], timing.get("duty", 0.74), timing.get("t0", t[0])
        method = "process period"
    else:
        z = np.linalg.svd(Bz[:: max(1, len(Bz) // 4000)], full_matrices=False)[2][0]
        period, strength = _period_from_oes(t, Bz @ z)
        duty0, t_ref = 0.74, t[0]
        method = "OES period"
        diag["diag|period_strength"] = strength
    is_sf6, edge, info = _phase_labels(t, Bz, period, duty0, t_ref)
    diag.update(info)
    diag["diag|period_s"] = float(period)
    if info["diag|phase_contrast"] < 1.5:                              # no clear two-phase structure
        method += ", phases unclear"
    diag["diag|phase_method"] = method
    clean = edge > edge_s
    jF = names.index("F_703.7")
    diag["diag|line_snr_F703"] = float(L[:, jF].mean() / (L[:, jF].std() / np.sqrt(len(L)) + 1e-12))
    # features
    f = {}
    n_all = len(t)
    groups = {"all": np.ones(n_all, bool), "sf6": is_sf6 & clean, "c4f8": ~is_sf6 & clean}
    for g, m in groups.items():
        if m.sum() < 50:
            continue
        T = tot[m].mean()
        f[f"oes|{g}|log_total"] = float(np.log(max(T, 1e-12)))
        bm = B[m].mean(0)
        for j in range(nb):
            f[f"oes|{g}|band{int(edges[j])}"] = float(bm[j] * npx_bin[j] / max(T, 1e-12))
        lm = L[m].mean(0)
        for j, name in enumerate(names):
            f[f"oes|{g}|{name}"] = float(lm[j] * npx / max(T, 1e-12))
        idx = np.flatnonzero(m)
        first, last = idx[idx < 0.2 * n_all], idx[idx >= 0.8 * n_all]
        if len(first) > 10 and len(last) > 10:
            f[f"oes|{g}|log_total|late-early"] = float(np.log(max(tot[last].mean(), 1e-12)) - np.log(max(tot[first].mean(), 1e-12)))
            for name in TREND_LINES:
                j = names.index(name)
                f[f"oes|{g}|{name}|late-early"] = float(L[last, j].mean() / max(tot[last].mean(), 1e-12) * npx
                                                        - L[first, j].mean() / max(tot[first].mean(), 1e-12) * npx)
    # folded band profile (for checking the phase split later) and the first 60 s of the band series
    ph = ((t - (timing or {}).get("t0", t[0])) / period) % 1.0
    pb = np.minimum((ph * 96).astype(int), 95)
    cnt = np.maximum(np.bincount(pb, minlength=96), 1)
    debug["folded_bands"] = np.stack([np.bincount(pb, weights=B[:, j], minlength=96) / cnt for j in range(nb)], 1).astype(np.float32)
    debug["band_edges_nm"] = edges[:-1].astype(np.float32)
    first60 = t < t[0] + 60
    debug["bands_first60s"] = B[first60].astype(np.float32)
    debug["t_first60s"] = (t[first60] - t[0]).astype(np.float32)
    debug["is_sf6_first60s"] = is_sf6[first60]
    return f, diag, debug


def oes_diagnostics_summary(diag):
    """One printed paragraph that tells whether the plasma-on and phase detection behaved on the real files."""
    if diag is None or not len(diag):
        return
    d = diag
    def col(c, fn, fmt):
        return fmt.format(fn(d[c])) if c in d and d[c].notna().any() else "n/a"
    print(f"OES checks over {len(d)} wafers: plasma-on fraction median {col('diag|on_fraction', np.nanmedian, '{:.3f}')}; "
          f"SF6-phase fraction median {col('diag|sf6_fraction', np.nanmedian, '{:.2f}')} (expected ~ the SF6 duty of the machine signals, ~0.72); "
          f"phase contrast median {col('diag|phase_contrast', np.nanmedian, '{:.1f}')} (>> 1 = clear phases); "
          f"period {col('diag|period_s', np.nanmedian, '{:.3f}')} s; sampling {col('diag|dt_median_s', np.nanmedian, '{:.3f}')} s")
    if "diag|phase_method" in d:
        print("  phase method:", d["diag|phase_method"].astype(str).value_counts().to_dict())
    if "diag|code_dtype" in d:
        print("  codes:", d["diag|code_dtype"].astype(str).value_counts().to_dict(),
              f"range {col('diag|code_min', np.nanmin, '{:.0f}')}..{col('diag|code_max', np.nanmax, '{:.0f}')}",
              "| codes beyond dictionary:", int(d["diag|codes_beyond_dictionary"].fillna(0).sum()) if "diag|codes_beyond_dictionary" in d else 0)


def oes_features(data_dir, keys=None, verbose=True, cache=None, timing=None, debug_path=None):
    """Per-wafer OES features from the daily files Day_YYYY_MM_DD.nc (one NetCDF group per wafer).

    Reads one wafer at a time. Results are cached as CSV if `cache` is set; per-wafer checks of the phase detection go
    to <cache>_diagnostics.csv and are printed (they are not used as features). `timing` = {key: {"period", "duty",
    "t0"}} from the process data. `debug_path`: small .npz with mean spectra and folded band profiles for checking."""
    dcache = None if not cache else cache.replace(".csv", "_diagnostics.csv")
    if cache and os.path.exists(cache):
        F = pd.read_csv(cache, index_col=0)
        if verbose:
            print(f"OES features loaded from cache {cache}: {F.shape}")
            if dcache and os.path.exists(dcache):
                oes_diagnostics_summary(pd.read_csv(dcache, index_col=0))
        return F
    from netCDF4 import Dataset
    files = [p for p in sorted(glob.glob(os.path.join(data_dir, "*Day_20*.nc"))) if "Dictionary" not in p]
    if not files:
        if verbose:
            print("no Day_*.nc OES files found - OES features skipped")
        return None
    dec = _decoder(data_dir, "oes_dict")
    if dec is None:
        warnings.warn("Dictionary_OES.nc not found - using raw OES codes instead of intensities")
    else:
        dec = np.asarray(dec, dtype=np.float32)
    rows, diags, dbg = {}, {}, {}
    for p in files:
        m = re.search(r"Day_(\d{4})_(\d{2})_(\d{2})", os.path.basename(p))
        date = f"{m.group(1)}-{m.group(2)}-{m.group(3)}"
        t0 = time.time()
        with Dataset(p, "r") as ds:
            for g in ds.groups:
                mw = re.search(r"(\d+)\s*$", g)
                if not mw:
                    continue
                key = f"{date}_{int(mw.group(1)):02d}"
                if keys is not None and key not in keys:
                    continue
                grp = ds.groups[g]
                spec = None
                try:
                    spec, cinfo = _decode_raw(grp["data"], dec)
                    wl = np.asarray(grp["wavelengths"][:], dtype=float)
                    if spec.shape[1] != len(wl) and spec.shape[0] == len(wl):
                        spec = np.ascontiguousarray(spec.T)
                    t = _epoch_s(grp["times"])
                    o = np.argsort(t, kind="stable")
                    if (np.diff(o) < 0).any():
                        spec, t = spec[o], t[o]
                    f, dg, db = _oes_wafer_features(spec, wl, t, None if timing is None else timing.get(key))
                    rows[key], diags[key] = f, {**cinfo, **dg}
                    dbg[key] = db
                except Exception as e:                                   # keep going; the wafer gets NaNs
                    warnings.warn(f"OES {os.path.basename(p)}/{g}: {type(e).__name__}: {e}")
                    rows[key], diags[key] = {}, {"diag|phase_method": f"error: {type(e).__name__}: {e}"}
                finally:
                    spec = None
        if verbose:
            print(f"  {os.path.basename(p)}: {sum(1 for k in rows if k.startswith(date))} wafers ({time.time() - t0:.0f} s)")
    F = pd.DataFrame.from_dict(rows, orient="index").sort_index()
    Dg = pd.DataFrame.from_dict(diags, orient="index").sort_index()
    if len(F):
        F = F.loc[:, F.notna().mean() >= 0.9]                            # allow a few missing wafers (imputed later)
        F = F.loc[:, F.std() > 0]
    if cache:
        F.to_csv(cache)
        Dg.to_csv(dcache)
    if debug_path and dbg:
        ks = sorted(dbg)
        out = {"keys": np.array(ks), "wavelengths": wl.astype(np.float32),
               "mean_spec": np.stack([dbg[k]["mean_spec"] for k in ks])}
        fb = [k for k in ks if "folded_bands" in dbg[k]]
        if fb:
            out["folded_keys"] = np.array(fb)
            out["folded_bands"] = np.stack([dbg[k]["folded_bands"] for k in fb])
            out["band_edges_nm"] = dbg[fb[0]]["band_edges_nm"]
            k0 = fb[0]
            for a in ("bands_first60s", "t_first60s", "is_sf6_first60s"):
                out[f"{a}_{k0}"] = dbg[k0][a]
        np.savez_compressed(debug_path, **out)
    if verbose:
        print(f"OES features: {F.shape[1]} for {F.shape[0]} wafers")
        oes_diagnostics_summary(Dg)
    return F


def context_features(meta):
    """Information known without any sensor: the wafer's position in its lot and the lot's conditioning recipe."""
    C = pd.DataFrame(index=meta.index)
    C["wafer_order"] = meta.wafer.astype(float)
    C["log_n_conditioning"] = np.log(meta.n_conditioning.astype(float))
    for s in ("chuck", "Si", "SiO2"):
        C[f"cond_on_{s}"] = (meta.cond_substrate == s).astype(float)
    return C

In [ ]:
# ===================================================================== core 3/3: priors, conditioning, site selection
ALPHAS = np.logspace(-2, 5, 29)


def nan_fill(X):
    """Column means ignoring NaN (0 for columns that are entirely missing) - used to impute missing features."""
    X = np.asarray(X, dtype=float)
    cnt = (~np.isnan(X)).sum(0)
    tot = np.where(np.isnan(X), 0.0, X).sum(0)
    return np.where(cnt > 0, tot / np.maximum(cnt, 1), 0.0)


def ridge_alpha_lolo(X, T, groups, alphas=ALPHAS):
    """Ridge penalty per target chosen by leave-one-lot-out inside the training lots (SVD path, fast)."""
    errs = np.zeros((len(alphas), T.shape[1]))
    for g in np.unique(groups):
        tr, te = groups != g, groups == g
        xm, tm = X[tr].mean(0), T[tr].mean(0)
        U, s, Vt = np.linalg.svd(X[tr] - xm, full_matrices=False)
        UtT = U.T @ (T[tr] - tm)
        Z = (X[te] - xm) @ Vt.T
        for i, a in enumerate(alphas):
            P = Z @ ((s / (s ** 2 + a))[:, None] * UtT) + tm
            errs[i] += ((P - T[te]) ** 2).sum(0)
    return alphas[np.argmin(errs, 0)]


def enet_offset(Xs, y, groups):
    """Elastic net for the wafer offset; penalty and L1 share chosen by leave-one-lot-out inside the training lots."""
    from sklearn.linear_model import ElasticNetCV
    from sklearn.model_selection import LeaveOneGroupOut
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        cv = list(LeaveOneGroupOut().split(Xs, y, groups))
        return ElasticNetCV(l1_ratio=[0.05, 0.3, 0.7, 1.0], n_alphas=25, cv=cv, max_iter=5000, tol=1e-3).fit(Xs, y)


class MapVM:
    """Virtual-metrology prior for a whole map: map = mean map + offset + sum_j score_j * shape_mode_j.

    The leading shape scores are ridge regressions on the features (penalty per target by leave-one-lot-out inside
    the training lots); the offset - most of the wafer-to-wafer variation - is an elastic net (OFFSET_MODEL =
    "enet", sparse: picks the few informative signals) or the same ridge ("ridge"). Without features the prior is
    the training mean map.
    """

    def __init__(self, n_modes=4, offset_model=None):
        self.n_modes = n_modes
        self.offset_model = offset_model or globals().get("OFFSET_MODEL", "enet")

    def fit(self, Y, X=None, groups=None):
        Y = np.asarray(Y, dtype=float)
        self.mu = Y.mean(0)
        off = Y.mean(1) - self.mu.mean()
        Dv = Y - self.mu - off[:, None]
        _, _, Vt = np.linalg.svd(Dv, full_matrices=False)
        self.modes = Vt[: self.n_modes]
        T = np.c_[off, Dv @ self.modes.T]
        self.t_mean = T.mean(0)
        self.W = None
        self.off = None
        if X is None or np.asarray(X).shape[1] == 0:
            return self
        X = np.asarray(X, dtype=float)
        self.x_fill = nan_fill(X)
        X = np.where(np.isnan(X), self.x_fill, X)
        self.x_mu, self.x_sd = X.mean(0), X.std(0) + 1e-12
        Xs = (X - self.x_mu) / self.x_sd
        alphas = ridge_alpha_lolo(Xs, T, groups) if groups is not None and len(np.unique(groups)) >= 3 \
            else np.full(T.shape[1], 100.0)
        U, s, Vt = np.linalg.svd(Xs - Xs.mean(0), full_matrices=False)
        UtT = U.T @ (T - self.t_mean)
        self.W = np.stack([Vt.T @ ((s / (s ** 2 + a)) * UtT[:, j]) for j, a in enumerate(alphas)], 1)
        self.xs_mean = Xs.mean(0)
        self.alphas = alphas
        if self.offset_model == "enet" and groups is not None and len(np.unique(groups)) >= 3:
            self.off = enet_offset(Xs, T[:, 0], groups)
        return self

    def predict_targets(self, X=None, n=1):
        if self.W is None:
            return np.tile(self.t_mean, (n if X is None else len(X), 1))
        X = np.asarray(X, dtype=float)
        X = np.where(np.isnan(X), self.x_fill, X)
        Xs = (X - self.x_mu) / self.x_sd
        T = (Xs - self.xs_mean) @ self.W + self.t_mean
        if self.off is not None:
            T[:, 0] = self.off.predict(Xs)
        return T

    def predict(self, X=None, n=1):
        T = self.predict_targets(X, n)
        return self.mu[None, :] + T[:, :1] + T[:, 1:] @ self.modes


def ridge_predict(Xtr, ytr, groups, Xte):
    """Single-target ridge regression (penalty by leave-one-lot-out), used for the difficulty model."""
    fill = nan_fill(Xtr)
    A = np.where(np.isnan(Xtr), fill, Xtr)
    B = np.where(np.isnan(Xte), fill, Xte)
    m, s = A.mean(0), A.std(0) + 1e-12
    A, B = (A - m) / s, (B - m) / s
    y = np.asarray(ytr, dtype=float)
    a = ridge_alpha_lolo(A, y[:, None], groups)[0] if len(np.unique(groups)) >= 3 else 100.0
    U, sv, Vt = np.linalg.svd(A, full_matrices=False)
    w = Vt.T @ ((sv / (sv ** 2 + a)) * (U.T @ (y - y.mean())))
    return B @ w + y.mean()


def oof_predict(Y, X, groups, n_modes=4):
    """Leave-one-lot-out predictions of MapVM for every training wafer (honest residuals for the covariance)."""
    P = np.zeros_like(Y, dtype=float)
    for g in np.unique(groups):
        tr, te = groups != g, groups == g
        vm = MapVM(n_modes).fit(Y[tr], None if X is None else X[tr], groups[tr])
        P[te] = vm.predict(None if X is None else X[te], n=int(te.sum()))
    return P


def residual_cov(R, nugget):
    """Second-moment matrix of out-of-fold residuals (bias included) plus a nugget on the diagonal."""
    R = np.asarray(R, dtype=float)
    return R.T @ R / len(R) + nugget * np.eye(R.shape[1])


def stationary_cov(R, xy, nugget_min=1e-6):
    """Classical geostatistics baseline: isotropic exponential covariance + nugget, fitted by least squares to the
    empirical covariance of the training deviations from the mean map (one sill, range and nugget for all sites)."""
    C = R.T @ R / len(R)
    d = np.hypot(xy[:, None, 0] - xy[None, :, 0], xy[:, None, 1] - xy[None, :, 1])
    off = ~np.eye(len(d), dtype=bool)
    best = None
    for ell in np.linspace(10, 400, 79):
        Kb = np.exp(-d / ell)
        s2 = max((Kb[off] * C[off]).sum() / (Kb[off] ** 2).sum(), 1e-9)
        nug = max(np.mean(np.diag(C)) - s2, nugget_min)
        err = ((s2 * Kb + nug * np.eye(len(d)) - C) ** 2).sum()
        if best is None or err < best[0]:
            best = (err, ell, s2, nug)
    _, ell, s2, nug = best
    return s2 * np.exp(-d / ell) + nug * np.eye(len(d))


def condition(mu, Sig, S, yS, noise_var):
    """Gaussian conditioning on noisy measurements yS at sites S: posterior mean and posterior variances."""
    S = list(S)
    if len(S) == 0:
        return mu.copy(), np.diag(Sig).copy()
    A = Sig[np.ix_(S, S)] + noise_var * np.eye(len(S))
    K = np.linalg.solve(A, Sig[S, :]).T
    mean = mu + K @ (np.asarray(yS, dtype=float) - mu[S])
    var = np.diag(Sig) - np.einsum("ij,ji->i", K, Sig[S, :])
    return mean, np.maximum(var, 1e-12)


def greedy_sites(Sig, k, noise_var, start=()):
    """Greedy A-optimal design: repeatedly add the site that most reduces the summed posterior variance."""
    C = Sig.copy()
    chosen = []
    for s in list(start):
        g = C[:, s] / (C[s, s] + noise_var)
        C = C - np.outer(g, C[s, :])
        chosen.append(s)
    while len(chosen) < k:
        gain = (C ** 2).sum(0) / (np.diag(C) + noise_var)
        gain[chosen] = -np.inf
        s = int(np.argmax(gain))
        g = C[:, s] / (C[s, s] + noise_var)
        C = C - np.outer(g, C[s, :])
        chosen.append(s)
    return chosen


def fsca_sites(Y, k):
    """Forward Selection Component Analysis (McLoone, Johnston & Susto, IEEE TASE 2018): greedily add the site
    whose least-squares reconstruction of all sites explains the most variance of the training maps."""
    Yc = Y - Y.mean(0)
    tot = (Yc ** 2).sum()
    chosen = []
    for _ in range(k):
        best, best_r2 = None, -np.inf
        for s in range(Y.shape[1]):
            if s in chosen:
                continue
            Xs = Yc[:, chosen + [s]]
            B, *_ = np.linalg.lstsq(Xs, Yc, rcond=None)
            r2 = 1 - ((Yc - Xs @ B) ** 2).sum() / tot
            if r2 > best_r2:
                best, best_r2 = s, r2
        chosen.append(best)
    return chosen


def qr_sites(Y, k):
    """QR column pivoting on the leading k PCA modes of the training maps (Manohar et al., 2018)."""
    from scipy.linalg import qr
    _, _, Vt = np.linalg.svd(Y - Y.mean(0), full_matrices=False)
    _, _, piv = qr(Vt[:max(k, 1)], pivoting=True)
    return [int(i) for i in piv[:k]]


def idw_predict(xy, S, yS, power=2.0):
    """Inverse-distance weighting from the measured sites (the interpolation the dataset authors used)."""
    S = list(S)
    yS = np.asarray(yS, dtype=float)
    d = np.hypot(xy[:, None, 0] - xy[None, S, 0], xy[:, None, 1] - xy[None, S, 1])
    out = np.empty(len(xy))
    for i in range(len(xy)):
        if (d[i] < 1e-9).any():
            out[i] = yS[np.argmin(d[i])]
        else:
            w = 1.0 / d[i] ** power
            out[i] = (w * yS).sum() / w.sum()
    return out


def zernike_basis(x, y, n_terms, radius=100.0):
    """First Zernike polynomials (Noll order) on the 200 mm wafer, a classical wafer-map basis."""
    rho, th = np.hypot(x, y) / radius, np.arctan2(y, x)
    t = [np.ones_like(rho), 2 * rho * np.cos(th), 2 * rho * np.sin(th), np.sqrt(3) * (2 * rho ** 2 - 1),
         np.sqrt(6) * rho ** 2 * np.sin(2 * th), np.sqrt(6) * rho ** 2 * np.cos(2 * th),
         np.sqrt(8) * (3 * rho ** 3 - 2 * rho) * np.sin(th), np.sqrt(8) * (3 * rho ** 3 - 2 * rho) * np.cos(th),
         np.sqrt(5) * (6 * rho ** 4 - 6 * rho ** 2 + 1)]
    return np.stack(t[:n_terms], 1)


def zernike_predict(xy, mu, S, yS, ridge=1e-3):
    """Deviation from the training mean map fitted with as many Zernike terms as there are sites (max 9)."""
    S = list(S)
    n = min(len(S), 9)
    Z = zernike_basis(xy[:, 0], xy[:, 1], n)
    A = Z[S]
    c = np.linalg.solve(A.T @ A + ridge * np.eye(n), A.T @ (np.asarray(yS) - mu[S]))
    return mu + Z @ c


def offset_predict(mu, S, yS):
    """Training mean map shifted by the average deviation at the measured sites."""
    S = list(S)
    return mu + (np.asarray(yS) - mu[S]).mean()


def conformal_quantile(scores, alpha):
    """Split-conformal quantile with the finite-sample correction."""
    s = np.sort(np.asarray(scores, dtype=float))
    n = len(s)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    return float(s[min(k, n) - 1]) if n else np.inf


def clean_outliers(Y, z_thresh=10.0, noise_var=1e-4):
    """Flag gross measurement errors and replace them by their conditional expectation given the wafer's other
    sites. A cell is flagged if its deviation from the site's typical shape is > z_thresh robust s.d.
    (median / MAD over wafers of the wafer-centred maps). Returns (cleaned Y, boolean flag matrix, robust z)."""
    Y = np.asarray(Y, dtype=float)
    shape = Y - Y.mean(1, keepdims=True)
    med = np.median(shape, 0)
    mad = 1.4826 * np.median(np.abs(shape - med), 0)
    Z = (shape - med) / np.maximum(mad, 1e-9)
    flag = np.abs(Z) > z_thresh if z_thresh else np.zeros_like(Y, dtype=bool)
    Yc = Y.copy()
    if flag.any():
        ok = ~flag.any(1)
        mu = Y[ok].mean(0)
        Sig = np.cov(Y[ok], rowvar=False) + noise_var * np.eye(Y.shape[1])
        for w in np.flatnonzero(flag.any(1)):
            S, T = np.flatnonzero(~flag[w]), np.flatnonzero(flag[w])
            Yc[w, T] = condition(mu, Sig, S, Y[w, S], noise_var)[0][T]
    return Yc, flag, Z


def paired(df_a, df_b, col="rmse", n_boot=2000, seed=0):
    """Paired comparison of two methods on the same wafers: mean over lots of the per-lot mean difference (a - b),
    95 % lot-bootstrap CI and the number of lots where a is better (lower)."""
    a = df_a.set_index("key")[[col, "lot"]]
    b = df_b.set_index("key")[[col]].rename(columns={col: col + "_b"})
    j = a.join(b, how="inner")
    d = (j[col] - j[col + "_b"]).groupby(j["lot"]).mean().to_numpy()
    rng = np.random.default_rng(seed)
    boots = d[rng.integers(0, len(d), size=(n_boot, len(d)))].mean(1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"mean_a": float(j[col].groupby(j["lot"]).mean().mean()), "mean_b": float(j[col + "_b"].groupby(j["lot"]).mean().mean()),
            "diff_a_minus_b": float(d.mean()), "ci_lo": float(lo), "ci_hi": float(hi),
            "lots_a_better": int((d < 0).sum()), "n_lots": int(len(d))}

In [ ]:
# ===================================================================== experiments (leave-one-lot-out)
METRICS = ["mean", "sd", "cv_pct", "range", "edge_minus_centre", "sd_inner", "cv_inner_pct"]


def score_maps(pred, truth, r):
    """Errors of predicted maps (rows) against the measured maps: RMSE on all / inner / edge sites and the
    absolute errors of the uniformity metrics."""
    pred, truth = np.atleast_2d(pred), np.atleast_2d(truth)
    e = pred - truth
    inner, edge = r <= R_INNER, r >= R_EDGE
    out = pd.DataFrame({"rmse": np.sqrt((e ** 2).mean(1)), "rmse_inner": np.sqrt((e[:, inner] ** 2).mean(1)),
                        "rmse_edge": np.sqrt((e[:, edge] ** 2).mean(1)), "max_abs": np.abs(e).max(1)})
    mp, mt = uniformity_metrics(pred, r), uniformity_metrics(truth, r)
    for m in METRICS:
        out[f"err_{m}"] = np.abs(mp[m].values - mt[m].values)
    return out


def _rows(tab, base):
    tab = tab.copy()
    for k, v in base.items():
        tab[k] = v
    return tab


def exp_site_selection(Y, lots, keys, xy, r, std_idx, K, noise_var, nugget, n_random=20, seed=0, edge_idx=None):
    """RQ1: which sites? Measurement-only prior (training mean map + residual covariance); designs learned on the
    training lots (FSCA, QR, greedy Bayesian) vs the dataset's standard layouts, the same layouts with the outer
    sites on the edge ring, and random sites. Estimators: the learned-covariance conditioning ("Kriging") and
    baselines without history (IDW) and with history (mean map + offset, mean map + IDW of the residuals,
    stationary kriging of the deviation from the mean map)."""
    rng = np.random.default_rng(seed)
    out = []
    for g in np.unique(lots):
        tr, te = lots != g, lots == g
        Ytr, Yte = Y[tr], Y[te]
        mu0 = Ytr.mean(0)
        Sig0 = residual_cov(Ytr - oof_predict(Ytr, None, lots[tr]), nugget)
        Sst = stationary_cov(Ytr - mu0, xy)
        designs = {"FSCA": fsca_sites(Ytr, K), "QR": qr_sites(Ytr, K), "Greedy": greedy_sites(Sig0, K, noise_var)}
        base = {"lot": g}

        def baselines(S):
            return (("IDW", lambda y: idw_predict(xy, S, y[S])),
                    ("Zernike", lambda y: zernike_predict(xy, mu0, S, y[S])),
                    ("Offset", lambda y: offset_predict(mu0, S, y[S])),
                    ("IDW-residual", lambda y: mu0 + idw_predict(xy, S, (y - mu0)[S])),
                    ("Stationary", lambda y: condition(mu0, Sst, S, y[S], noise_var)[0]))
        for k in range(K + 1):
            for dname, D_ in designs.items():
                S = D_[:k]
                P = np.stack([condition(mu0, Sig0, S, y[S], noise_var)[0] for y in Yte])
                out.append(_rows(score_maps(P, Yte, r), {**base, "key": keys[te], "design": dname,
                                                         "estimator": "Kriging", "k": k}))
                if dname == "FSCA" and k >= 1:
                    for ename, f in baselines(S):
                        P = np.stack([f(y) for y in Yte])
                        out.append(_rows(score_maps(P, Yte, r), {**base, "key": keys[te], "design": dname,
                                                                 "estimator": ename, "k": k}))
            if k >= 1:                                                   # random designs, averaged over draws
                acc = None
                for _ in range(n_random):
                    S = list(rng.permutation(Y.shape[1])[:k])
                    P = np.stack([condition(mu0, Sig0, S, y[S], noise_var)[0] for y in Yte])
                    sc = score_maps(P, Yte, r)
                    acc = sc if acc is None else acc + sc
                out.append(_rows(acc / n_random, {**base, "key": keys[te], "design": "Random",
                                                  "estimator": "Kriging", "k": k}))
        fixed = [("Standard", std_idx)] + ([("Edge-inclusive", edge_idx)] if edge_idx else [])
        for dname, layouts in fixed:
            for k, S in layouts.items():
                for ename, f in (("Kriging", lambda y: condition(mu0, Sig0, S, y[S], noise_var)[0]),) + baselines(S):
                    P = np.stack([f(y) for y in Yte])
                    out.append(_rows(score_maps(P, Yte, r), {**base, "key": keys[te], "design": dname,
                                                             "estimator": ename, "k": k}))
    return pd.concat(out, ignore_index=True)


def exp_information(Y, lots, keys, r, levels, std_idx, K, noise_var, nugget, n_modes=4):
    """RQ2: how much is process information worth? For each information level the prior mean comes from MapVM
    on that level's features and the covariance from its out-of-fold residuals; sites from greedy design under
    that covariance, from FSCA (measurement-only design) and from the standard layouts."""
    out, alphas = [], []
    for g in np.unique(lots):
        tr, te = lots != g, lots == g
        Ytr, Yte = Y[tr], Y[te]
        fsca = fsca_sites(Ytr, K)
        for lname, X in levels.items():
            Xtr = None if X is None else X[tr]
            Xte = None if X is None else X[te]
            vm = MapVM(n_modes).fit(Ytr, Xtr, lots[tr])
            if vm.W is not None:
                alphas.append({"lot": g, "level": lname, **{f"alpha_{j}": a for j, a in enumerate(vm.alphas)}})
            Sig = residual_cov(Ytr - oof_predict(Ytr, Xtr, lots[tr], n_modes), nugget)
            mu = vm.predict(Xte, n=int(te.sum()))
            designs = {"Greedy": greedy_sites(Sig, K, noise_var), "FSCA": fsca}
            for dname, D_ in designs.items():
                for k in range(K + 1):
                    S = D_[:k]
                    P = np.stack([condition(mu[i], Sig, S, Yte[i, S], noise_var)[0] for i in range(len(Yte))])
                    out.append(_rows(score_maps(P, Yte, r), {"lot": g, "key": keys[te], "level": lname,
                                                             "design": dname, "k": k}))
            for k, S in std_idx.items():
                P = np.stack([condition(mu[i], Sig, S, Yte[i, S], noise_var)[0] for i in range(len(Yte))])
                out.append(_rows(score_maps(P, Yte, r), {"lot": g, "key": keys[te], "level": lname,
                                                         "design": "Standard", "k": k}))
    return pd.concat(out, ignore_index=True), pd.DataFrame(alphas)


def exp_adaptive(Y, lots, keys, r, levels, K, noise_var, nugget, n_modes=4, n_neighbors=25, blend=0.5):
    """RQ3: can process information choose the sites per wafer? Static greedy design under the level's residual
    covariance vs a wafer-specific design under a covariance blended with that of the wafer's nearest
    training wafers in feature space."""
    out = []
    for g in np.unique(lots):
        tr, te = lots != g, lots == g
        Ytr, Yte = Y[tr], Y[te]
        for lname, X in levels.items():
            if X is None:
                continue
            Xtr, Xte = X[tr], X[te]
            vm = MapVM(n_modes).fit(Ytr, Xtr, lots[tr])
            R = Ytr - oof_predict(Ytr, Xtr, lots[tr], n_modes)
            Sig = residual_cov(R, nugget)
            mu = vm.predict(Xte)
            static = greedy_sites(Sig, K, noise_var)
            fill = nan_fill(Xtr)
            A = np.where(np.isnan(Xtr), fill, Xtr)
            B = np.where(np.isnan(Xte), fill, Xte)
            m_, s_ = A.mean(0), A.std(0) + 1e-12
            A, B = (A - m_) / s_, (B - m_) / s_
            for i in range(len(Yte)):
                nn = np.argsort(((A - B[i]) ** 2).sum(1))[:n_neighbors]
                Sx = (1 - blend) * Sig + blend * residual_cov(R[nn], nugget)
                adapt = greedy_sites(Sx, K, noise_var)
                for k in range(1, K + 1):
                    for dname, S, C in (("Static", static[:k], Sig), ("Adaptive", adapt[:k], Sx)):
                        p = condition(mu[i], C, S, Yte[i, S], noise_var)[0]
                        out.append(_rows(score_maps(p, Yte[i], r), {"lot": g, "key": keys[te][i], "level": lname,
                                                                    "design": dname, "k": k}))
    return pd.concat(out, ignore_index=True)


def exp_calibration(Y, lots, keys, r, levels, ks, noise_var, nugget, alpha=0.1, n_modes=4, design="Greedy"):
    """RQ4: are the reported uncertainties trustworthy on unseen lots? Gaussian posterior intervals vs split-
    conformal intervals calibrated on the training lots (leave-one-lot-out), for every site and for the
    wafer-level uniformity metrics."""
    z = 1.6448536269514722                                                 # 90 % Gaussian
    out = []
    Kmax = max(ks)
    for g in np.unique(lots):
        tr, te = lots != g, lots == g
        Ytr, Yte, ltr = Y[tr], Y[te], lots[tr]
        for lname, X in levels.items():
            Xtr = None if X is None else X[tr]
            Xte = None if X is None else X[te]
            Poof = oof_predict(Ytr, Xtr, ltr, n_modes)
            R = Ytr - Poof
            site_sc = {k: [] for k in ks}
            met_err = {k: {m: [] for m in METRICS} for k in ks}
            for g2 in np.unique(ltr):                                     # calibration: each training lot in turn
                m2 = ltr != g2
                S2 = residual_cov(R[m2], nugget)
                d2 = greedy_sites(S2, Kmax, noise_var) if design == "Greedy" else fsca_sites(Ytr[m2], Kmax)
                for i in np.flatnonzero(~m2):
                    for k in ks:
                        mean, var = condition(Poof[i], S2, d2[:k], Ytr[i, d2[:k]], noise_var)
                        site_sc[k].extend(np.abs(mean - Ytr[i]) / np.sqrt(var))
                        mp, mt = uniformity_metrics(mean, r), uniformity_metrics(Ytr[i], r)
                        for m in METRICS:
                            met_err[k][m].append(abs(mp[m].iloc[0] - mt[m].iloc[0]))
            vm = MapVM(n_modes).fit(Ytr, Xtr, ltr)
            mu = vm.predict(Xte, n=int(te.sum()))
            Sig = residual_cov(R, nugget)
            d = greedy_sites(Sig, Kmax, noise_var) if design == "Greedy" else fsca_sites(Ytr, Kmax)
            for k in ks:
                q = conformal_quantile(site_sc[k], alpha)
                qm = {m: conformal_quantile(met_err[k][m], alpha) for m in METRICS}
                for i in range(len(Yte)):
                    S = d[:k]
                    mean, var = condition(mu[i], Sig, S, Yte[i, S], noise_var)
                    e = np.abs(mean - Yte[i])
                    unm = np.setdiff1d(np.arange(Y.shape[1]), S)          # coverage on the unmeasured sites
                    mp, mt = uniformity_metrics(mean, r), uniformity_metrics(Yte[i], r)
                    row = {"lot": g, "key": keys[te][i], "level": lname, "k": k,
                           "cov_site_gauss": float((e[unm] <= z * np.sqrt(var[unm])).mean()),
                           "cov_site_conformal": float((e[unm] <= q * np.sqrt(var[unm])).mean()),
                           "halfwidth_site_gauss": float(np.mean(z * np.sqrt(var[unm]))),
                           "halfwidth_site_conformal": float(np.mean(q * np.sqrt(var[unm]))),
                           "mean_z2": float(np.mean(e[unm] ** 2 / var[unm]))}
                    for m in METRICS:
                        row[f"cov_{m}"] = float(abs(mp[m].iloc[0] - mt[m].iloc[0]) <= qm[m])
                        row[f"halfwidth_{m}"] = qm[m]
                    out.append(row)
    return pd.DataFrame(out)


def exp_measure_or_skip(Y, lots, keys, r, levels, taus, noise_var, nugget, alpha=0.1, n_modes=4):
    """RQ5: can process information decide which wafers need a measurement at all? Normalised conformal interval
    for the wafer-mean etch depth from the virtual-metrology prior (difficulty model on the same features);
    a wafer skips metrology if the half-width is below tau, otherwise it gets the first FSCA site."""
    out = []
    for g in np.unique(lots):
        tr, te = lots != g, lots == g
        Ytr, Yte, ltr = Y[tr], Y[te], lots[tr]
        first_site = fsca_sites(Ytr, 1)
        for lname, X in levels.items():
            Xtr = None if X is None else X[tr]
            Xte = None if X is None else X[te]
            Poof = oof_predict(Ytr, Xtr, ltr, n_modes)
            res_mean = np.abs(Ytr.mean(1) - Poof.mean(1))
            Sig = residual_cov(Ytr - Poof, nugget)
            if X is None:                                                 # no features: one width for everyone
                d_oof, d_te = np.ones(len(Ytr)), np.ones(int(te.sum()))
            else:                                                         # difficulty model, cross-fitted
                logabs = np.log(res_mean + 1e-3)
                d_oof = np.zeros(len(Ytr))
                for g2 in np.unique(ltr):
                    m2 = ltr != g2
                    d_oof[~m2] = np.exp(ridge_predict(Xtr[m2], logabs[m2], ltr[m2], Xtr[~m2]))
                d_te = np.exp(ridge_predict(Xtr, logabs, ltr, Xte))
            q = conformal_quantile(res_mean / d_oof, alpha)
            vm = MapVM(n_modes).fit(Ytr, Xtr, ltr)
            mu = vm.predict(Xte, n=int(te.sum()))
            for i in range(len(Yte)):
                p1 = condition(mu[i], Sig, first_site, Yte[i, first_site], noise_var)[0]
                for tau in taus:
                    skip = q * d_te[i] <= tau
                    p = mu[i] if skip else p1
                    out.append({"lot": g, "key": keys[te][i], "level": lname, "tau": tau, "skipped": float(skip),
                                "err_mean": abs(p.mean() - Yte[i].mean()),
                                "within_tau": float(abs(p.mean() - Yte[i].mean()) <= tau),
                                "rmse": float(np.sqrt(((p - Yte[i]) ** 2).mean())), "halfwidth": q * d_te[i]})
    return pd.DataFrame(out)


def exp_inline(D, Y, lots, keys, r, levels, noise_var, nugget, value="si_etch", n_modes=4):
    """RQ6: reconstruct the offline 89-point map from the real in-line 9-point measurements (other instrument,
    measured on the process day). The grid symmetry and per-site offsets between the campaigns are learned on
    the training lots only."""
    d9, d89, sites = D["d9"], D["d89"], D["sites"]
    lab2i = {l: i for i, l in enumerate(sites.label)}
    xy2lab = {(int(round(x * 1000)), int(round(y * 1000))): l for l, x, y in zip(sites.label, sites.x, sites.y)}
    out, chosen = [], []
    for g in np.unique(lots):
        tr, te = lots != g, lots == g
        te_keys = [k for k in keys[te] if (d9.experiment_key == k).sum() == 9]
        if not te_keys:
            continue
        trk = set(keys[tr])
        aud = audit_campaigns(d9, d89, value, keys=trk)
        tf = aud["transform"].iloc[0]
        chosen.append({"lot": g, "transform": tf})
        kidx = {k: i for i, k in enumerate(keys)}
        maps9 = {}
        for orient in dict.fromkeys([tf, "identity"]):                   # learned orientation and the files as given
            f = D4[orient]
            V9 = {}
            for _, row in d9.iterrows():
                x2, y2 = f(row.X, row.Y)
                V9.setdefault(row.experiment_key, {})[xy2lab[(int(x2), int(y2))]] = row[value]
            labs = sorted({l for v in V9.values() for l in v}, key=lambda l: lab2i[l])
            ktr = [k for k in keys[tr] if k in V9 and len(V9[k]) == len(labs)]
            off = np.mean([[V9[k][l] - Y[kidx[k], lab2i[l]] for l in labs] for k in ktr], axis=0)
            maps9[orient] = (V9, labs, [lab2i[l] for l in labs], off)
        for lname, X in levels.items():
            Xtr = None if X is None else X[tr]
            vm = MapVM(n_modes).fit(Y[tr], Xtr, lots[tr])
            Sig = residual_cov(Y[tr] - oof_predict(Y[tr], Xtr, lots[tr], n_modes), nugget)
            for k in te_keys:
                i = kidx[k]
                mu = vm.predict(None if X is None else X[i:i + 1], n=1)[0]
                V9, labs, S, off = maps9[tf]
                meas = np.array([V9[k][l] for l in labs])
                V9i, labsi, Si, offi = maps9["identity"]
                measi = np.array([V9i[k][l] for l in labsi])
                cases = ((f"in-line, {tf} + site offsets", S, meas - off),
                         (f"in-line, {tf}, no offsets", S, meas),
                         ("in-line, coordinates as given + site offsets", Si, measi - offi),
                         ("offline values at the same sites", S, Y[i, S]))
                for case, SS, yS in cases:
                    p = condition(mu, Sig, SS, yS, noise_var)[0]
                    out.append(_rows(score_maps(p, Y[i], r), {"lot": g, "key": k, "level": lname, "case": case}))
    return pd.concat(out, ignore_index=True), pd.DataFrame(chosen)


def exp_forward(Y, lots, keys, r, std_idx, edge_idx, levels, noise_var, nugget, n_modes=4, ks=(0, 1, 2, 3, 5, 9),
                first_test=3):
    """Robustness: forward in time. Lots are numbered in date order; lot j is predicted from lots 1..j-1 only (as a
    fab would deploy the model), for the main designs and information levels."""
    out = []
    for j in np.unique(lots):
        if j < first_test:
            continue
        tr, te = lots < j, lots == j
        Ytr, Yte = Y[tr], Y[te]
        fs = fsca_sites(Ytr, max(ks))
        for lname, X in levels.items():
            Xtr, Xte = (None, None) if X is None else (X[tr], X[te])
            vm = MapVM(n_modes).fit(Ytr, Xtr, lots[tr])
            Sig = residual_cov(Ytr - oof_predict(Ytr, Xtr, lots[tr], n_modes), nugget)
            mu = vm.predict(Xte, n=int(te.sum()))
            designs = [("FSCA", k, fs[:k]) for k in ks] + [("Standard", 9, std_idx[9])]
            if edge_idx:
                designs.append(("Edge-inclusive", 9, edge_idx[9]))
            for dname, k, S in designs:
                P = np.stack([condition(mu[i], Sig, S, Yte[i, S], noise_var)[0] for i in range(len(Yte))])
                out.append(_rows(score_maps(P, Yte, r), {"lot": j, "key": keys[te], "level": lname, "design": dname,
                                                         "k": k, "n_training_lots": int(len(np.unique(lots[tr])))}))
    return pd.concat(out, ignore_index=True)


def uniformity_estimates(Y, lots, r, std_idx, X_best, noise_var, nugget, metric="cv_pct", n_modes=4):
    """Signed error (estimate - truth) of a wafer's uniformity metric on unseen lots for several strategies:
    the standard 9 points taken at face value, the standard 9 points with the learned spatial model, 1 and 3
    learned sites, and virtual metrology without any measurement."""
    rows = []
    for g in np.unique(lots):
        tr, te = lots != g, lots == g
        Ytr, Yte = Y[tr], Y[te]
        mu0 = Ytr.mean(0)
        Sig0 = residual_cov(Ytr - oof_predict(Ytr, None, lots[tr], n_modes), nugget)
        fs = fsca_sites(Ytr, 3)
        S9 = std_idx[9]
        if X_best is not None:
            vm = MapVM(n_modes).fit(Ytr, X_best[tr], lots[tr])
            vm_pred = vm.predict(X_best[te])
        for i, y in enumerate(Yte):
            truth = uniformity_metrics(y, r)[metric].iloc[0]
            # the plain 9-point statistic (no model): s.d./mean of the 9 values
            v9 = y[S9]
            naive = {"cv_pct": 100 * v9.std() / v9.mean(), "sd": v9.std(), "range": np.ptp(v9)}.get(metric, np.nan)
            est = {"9 standard,\nas measured": naive,
                   "9 standard\n+ model": uniformity_metrics(condition(mu0, Sig0, S9, y[S9], noise_var)[0], r)[metric].iloc[0],
                   "3 learned\n+ model": uniformity_metrics(condition(mu0, Sig0, fs, y[fs], noise_var)[0], r)[metric].iloc[0],
                   "1 learned\n+ model": uniformity_metrics(condition(mu0, Sig0, fs[:1], y[fs[:1]], noise_var)[0], r)[metric].iloc[0]}
            est["0 sites,\nmean map"] = uniformity_metrics(mu0, r)[metric].iloc[0]
            if X_best is not None:
                est["0 sites,\nVM only"] = uniformity_metrics(vm_pred[i], r)[metric].iloc[0]
            rows.append({"lot": g, **{k: v - truth for k, v in est.items()}})
    return pd.DataFrame(rows)


# ===================================================================== summaries
def macro(df, by, cols, n_boot=2000, seed=0):
    """Mean over lots of the per-lot means (each held-out lot counts once), with a lot-bootstrap 95 % CI."""
    rng = np.random.default_rng(seed)
    lot_means = df.groupby(by + ["lot"])[cols].mean().reset_index()
    rows = []
    for key, d in lot_means.groupby(by):
        key = key if isinstance(key, tuple) else (key,)
        vals = d[cols].to_numpy()
        boots = vals[rng.integers(0, len(vals), size=(n_boot, len(vals)))].mean(1)
        row = dict(zip(by, key))
        for j, c in enumerate(cols):
            row[c] = vals[:, j].mean()
            row[c + "_lo"], row[c + "_hi"] = np.percentile(boots[:, j], [2.5, 97.5])
        row["n_lots"] = len(vals)
        rows.append(row)
    return pd.DataFrame(rows)


def sites_to_match(curve_ref, target):
    """Smallest (linearly interpolated) number of sites at which a reference error curve reaches `target`."""
    ks, ys = np.asarray(curve_ref.index, dtype=float), np.asarray(curve_ref.values, dtype=float)
    if ys[0] <= target:
        return 0.0
    for j in range(1, len(ks)):
        if ys[j] <= target:
            return float(ks[j - 1] + (ys[j - 1] - target) / (ys[j - 1] - ys[j]) * (ks[j] - ks[j - 1]))
    return np.inf

In [ ]:
# ===================================================================== figures and tables for the paper
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

INK, INK2, GRID, MUTED = "#0b0b0b", "#52514e", "#e3e2de", "#b8b7b2"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]
SEQ_BLUE = LinearSegmentedColormap.from_list("seq_blue", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf",
                                                          "#184f95", "#0d366b"])
SEQ_ORANGE = LinearSegmentedColormap.from_list("seq_orange", ["#fde3d6", "#f7b393", "#eb6834", "#b84a1f", "#7a2e10"])
DIVERGING = LinearSegmentedColormap.from_list("div", ["#184f95", "#6da7ec", "#f0efec", "#ef9a99", "#a8302f"])

plt.rcParams.update({
    "font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8, "legend.fontsize": 7, "xtick.labelsize": 7,
    "ytick.labelsize": 7, "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.linewidth": 0.6, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1.4, "lines.markersize": 4,
    "legend.frameon": False, "figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight",
    "pdf.fonttype": 42, "ps.fonttype": 42, "axes.titleweight": "normal", "axes.titlelocation": "left",
})


def _save(fig, out_dir, name):
    os.makedirs(os.path.join(out_dir, "figures"), exist_ok=True)
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(out_dir, "figures", f"{name}.{ext}"))
    plt.show()
    plt.close(fig)


def _wafer_axes(ax):
    th = np.linspace(0, 2 * np.pi, 400)
    ax.plot(100 * np.cos(th), 100 * np.sin(th), color=INK2, lw=0.6)
    ax.set_aspect("equal")
    ax.set_xlim(-108, 108)
    ax.set_ylim(-108, 108)
    ax.set_xticks([-95, 0, 95])
    ax.set_yticks([-95, 0, 95])
    ax.grid(False)
    ax.set_xlabel("x (mm)")


def fig_wafer(sites, Y, std_labels, learned_labels, out_dir, unit="um"):
    """(a) mean etch-depth map with the standard and learned layouts, (b) radial profile, (c) wafer-to-wafer
    variation of the shape (after removing each wafer's mean)."""
    lab = list(sites.label)
    x, y, r = sites.x.values, sites.y.values, sites.r.values
    mean = Y.mean(0)
    shape_sd = (Y - Y.mean(1, keepdims=True)).std(0)
    fig, axs = plt.subplots(1, 3, figsize=(7.16, 2.45), gridspec_kw={"width_ratios": [1, 1.25, 1]})
    ax = axs[0]
    sc = ax.scatter(x, y, c=mean, cmap=SEQ_BLUE, marker="s", s=34, linewidths=0)
    si = [lab.index(l) for l in std_labels]
    ax.scatter(x[si], y[si], facecolors="none", edgecolors=INK, s=70, linewidths=0.8, label="standard 9-point")
    li = [lab.index(l) for l in learned_labels]
    ax.scatter(x[li], y[li], marker="^", color=SERIES[1], s=36, linewidths=0.6, edgecolors="white",
               label=f"learned, first {len(li)}")
    for n, j in enumerate(li, 1):
        ax.annotate(str(n), (x[j], y[j]), xytext=(-9 if x[j] > 60 else 4, 3), textcoords="offset points", fontsize=6,
                    color=INK)
    _wafer_axes(ax)
    ax.set_ylabel("y (mm)")
    ax.set_title("(a) mean etch depth")
    cb = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.02)
    cb.set_label(f"depth ({unit})")
    cb.outline.set_linewidth(0.4)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.3), ncol=1, handletextpad=0.3)
    ax = axs[1]
    sd = Y.std(0)
    ax.axvspan(R_EDGE, 100, color=GRID, lw=0, label=f"edge ring (r ≥ {R_EDGE:.0f} mm)")
    ax.errorbar(r, mean, yerr=sd, fmt="o", ms=3, color=SERIES[0], ecolor=MUTED, elinewidth=0.6, capsize=0,
                label="site mean ± s.d.")
    ax.scatter(r[si], mean[si], facecolors="none", edgecolors=INK, s=40, linewidths=0.8, zorder=3,
               label="standard 9-point")
    ax.set_xlabel("radius (mm)")
    ax.set_ylabel(f"etch depth ({unit})")
    ax.set_title("(b) radial profile")
    ax.legend(loc="upper left", fontsize=6.5)
    ax = axs[2]
    sc = ax.scatter(x, y, c=shape_sd, cmap=SEQ_ORANGE, marker="s", s=34, linewidths=0,
                    vmin=0, vmax=float(np.percentile(shape_sd, 98)))
    _wafer_axes(ax)
    ax.set_yticklabels([])
    ax.set_title("(c) wafer-to-wafer s.d. of shape")
    cb = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.02)
    cb.set_label(f"s.d. ({unit})")
    cb.outline.set_linewidth(0.4)
    fig.tight_layout()
    _save(fig, out_dir, "fig1_wafer_layouts")


def fig_audit(audit_tab, pairs_given, pairs_best, best_name, out_dir, value_name="Si etch depth"):
    """(a) agreement of the two campaigns under each grid symmetry, (b) matched values as given vs aligned."""
    fig, axs = plt.subplots(1, 2, figsize=(7.16, 2.7), gridspec_kw={"width_ratios": [1.1, 1]})
    ax = axs[0]
    t = audit_tab.sort_values("resid_sd_after_wafer_offset_um")
    cols = [SERIES[0] if n == best_name else MUTED for n in t["transform"]]
    ax.barh(np.arange(len(t))[::-1], t.resid_sd_after_wafer_offset_um, color=cols, height=0.6)
    ax.set_yticks(np.arange(len(t))[::-1])
    ax.set_yticklabels(t["transform"])
    ax.set_xlabel("step-height residual s.d. after per-wafer offset (um)")
    ax.set_title("(a) which grid symmetry aligns the campaigns?")
    ax.grid(axis="y", visible=False)
    for yy, v in zip(np.arange(len(t))[::-1], t.resid_sd_after_wafer_offset_um):
        ax.text(v, yy, f" {v:.3f}", va="center", fontsize=6.5, color=INK2)
    ax = axs[1]
    ax.scatter(pairs_given.v89, pairs_given.v9, s=8, color=MUTED, linewidths=0, label="coordinates as given")
    ax.scatter(pairs_best.v89, pairs_best.v9, s=8, color=SERIES[0], linewidths=0, label=f"after {best_name}")
    lo = min(pairs_given.v89.min(), pairs_given.v9.min())
    hi = max(pairs_given.v89.max(), pairs_given.v9.max())
    off = (pairs_best.v9 - pairs_best.v89).mean()
    ax.plot([lo, hi], [lo + off, hi + off], color=INK2, lw=0.8, ls="--", label=f"offset {off:+.2f} um")
    ax.set_xlabel(f"offline 89-point {value_name} (um)")
    ax.set_ylabel(f"in-line 9-point {value_name} (um)")
    ax.set_title("(b) matched sites")
    ax.legend(loc="upper center", bbox_to_anchor=(0.45, -0.27), ncol=3, frameon=False, handletextpad=0.3, columnspacing=1.0)
    fig.tight_layout()
    _save(fig, out_dir, "fig2_campaign_audit")


def fig_drift(Y, meta, out_dir):
    """Wafer-mean etch depth against the processing order inside each lot."""
    fig, ax = plt.subplots(figsize=(3.5, 2.3))
    m = Y.mean(1)
    for g in np.unique(meta.lot.values):
        s = meta.lot.values == g
        o = meta.wafer.values[s]
        ax.plot(o[np.argsort(o)], m[s][np.argsort(o)], color=MUTED, lw=0.7)
    agg = pd.Series(m).groupby(meta.wafer.values).mean()
    ax.plot(agg.index, agg.values, color=SERIES[0], marker="o", label="mean over lots")
    ax.plot([], [], color=MUTED, lw=0.7, label="one lot")
    ax.set_xlabel("wafer order in lot")
    ax.set_ylabel("wafer-mean etch depth (um)")
    ax.set_xticks(range(1, 11))
    ax.legend(loc="upper right")
    fig.tight_layout()
    _save(fig, out_dir, "fig3_drift")


def _curve(ax, d, x, y, label, color, marker, ls="-", band=True):
    d = d.sort_values(x)
    ax.plot(d[x], d[y], color=color, marker=marker, ls=ls, label=label, markeredgecolor="white", markeredgewidth=0.6)
    if band and f"{y}_lo" in d:
        ax.fill_between(d[x], d[f"{y}_lo"], d[f"{y}_hi"], color=color, alpha=0.10, lw=0)


def fig_curves(site_macro, info_macro, levels_order, out_dir, metric="rmse", ylabel="map RMSE (um)", design_b="FSCA"):
    """(a) error vs number of sites for the site-selection designs, (b) for the information levels."""
    fig, axs = plt.subplots(1, 2, figsize=(7.16, 2.5), sharey=True)
    ax = axs[0]
    kr = site_macro[site_macro.estimator == "Kriging"]
    for j, dname in enumerate(["FSCA", "Greedy", "QR", "Random"]):
        d = kr[kr.design == dname]
        if len(d):
            _curve(ax, d, "k", metric, dname, SERIES[j], MARKERS[j], band=(j == 0))
    d = kr[kr.design == "Standard"]
    ax.plot(d.k, d[metric], color=SERIES[4], marker=MARKERS[4], ls=":", label="standard 1/5/9",
            markeredgecolor="white", markeredgewidth=0.6)
    d = kr[kr.design == "Edge-inclusive"]
    if len(d):
        ax.plot(d.k, d[metric], color=SERIES[5], marker="P", ls=":", label="edge-inclusive 5/9",
                markeredgecolor="white", markeredgewidth=0.6)
    ax.set_xlabel("measured sites k")
    ax.set_ylabel(ylabel)
    ax.set_title("(a) where to measure (no process data)")
    ax.legend(loc="upper right")
    ax = axs[1]
    g = info_macro[info_macro.design == design_b]
    for j, lname in enumerate(levels_order):
        d = g[g.level == lname]
        if len(d):
            _curve(ax, d, "k", metric, lname, SERIES[j], MARKERS[j], band=(j == 0))
    ax.set_xlabel(f"measured sites k ({design_b} sites for every level)")
    ax.set_title("(b) what process information adds")
    ax.legend(loc="upper right")
    for ax in axs:
        ax.set_xticks(range(0, int(site_macro.k.max()) + 1, 2))
    fig.tight_layout()
    _save(fig, out_dir, f"fig4_error_vs_sites_{metric}")


def fig_uniformity(est, out_dir, metric="cv_pct", label="within-wafer CV (%)"):
    """Error of the wafer's uniformity metric for several measurement strategies (one dot per wafer):
    (a) including the plain 9-point statistic, (b) zoom on the model-based estimates."""
    fig, axs = plt.subplots(1, 2, figsize=(7.16, 2.5), gridspec_kw={"width_ratios": [1, 1.6]})
    names = list(est.columns)
    for ax, sub, title in ((axs[0], names[:2], "(a) 9-point statistic vs model"),
                           (axs[1], names[1:], "(b) model-based estimates (zoom)")):
        for j, n in enumerate(sub):
            v = est[n].dropna().values
            xj = j + (np.random.default_rng(j).random(len(v)) - 0.5) * 0.35
            ax.scatter(xj, v, s=7, color=MUTED if n == names[0] else SERIES[0], linewidths=0, alpha=0.8)
            ax.plot([j - 0.25, j + 0.25], [np.median(v)] * 2, color=INK, lw=1.0)
        ax.axhline(0, color=INK2, lw=0.6)
        ax.set_xticks(range(len(sub)))
        ax.set_xticklabels(sub, rotation=0, fontsize=6.5)
        ax.grid(axis="x", visible=False)
        ax.set_title(title)
    axs[0].set_ylabel(f"estimated − true {label}")
    fig.tight_layout()
    _save(fig, out_dir, f"fig5_uniformity_{metric}")


def fig_calibration(cal_macro, levels_show, out_dir, nominal=0.9):
    """Coverage of 90 % intervals on unseen lots: Gaussian posterior vs split-conformal."""
    fig, axs = plt.subplots(1, 2, figsize=(7.16, 2.3))
    for ax, (col, title) in zip(axs, (("site", "(a) unmeasured sites"), ("cv_pct", "(b) within-wafer CV"))):
        j = 0
        for lname in levels_show:
            d = cal_macro[cal_macro.level == lname].sort_values("k")
            if not len(d):
                continue
            if col == "site":
                ax.plot(d.k, d.cov_site_gauss, color=SERIES[j], marker=MARKERS[j], ls="--", label=f"{lname}, Gaussian",
                        markeredgecolor="white", markeredgewidth=0.6)
                ax.plot(d.k, d.cov_site_conformal, color=SERIES[j], marker=MARKERS[j], label=f"{lname}, conformal",
                        markeredgecolor="white", markeredgewidth=0.6)
            else:
                ax.plot(d.k, d[f"cov_{col}"], color=SERIES[j], marker=MARKERS[j], label=f"{lname}, conformal",
                        markeredgecolor="white", markeredgewidth=0.6)
            j += 1
        ax.axhline(nominal, color=INK2, lw=0.8, ls=":")
        ax.set_ylim(0.5, 1.02)
        ax.set_xlabel("measured sites k")
        ax.set_ylabel("coverage of 90 % intervals")
        ax.set_title(title)
        ax.legend(loc="lower right", ncol=1)
    fig.tight_layout()
    _save(fig, out_dir, "fig6_calibration")


def latex_table(df, path, caption, label, floatfmt="{:.3f}"):
    """Minimal booktabs LaTeX table (no extra dependencies)."""
    cols = list(df.columns)
    lines = ["\\begin{table}[t]", "\\centering", f"\\caption{{{caption}}}", f"\\label{{{label}}}",
             "\\footnotesize", "\\begin{tabular}{" + "l" * (df.index.nlevels) + "r" * len(cols) + "}", "\\toprule"]
    idx_names = [n or "" for n in (df.index.names if df.index.nlevels > 1 else [df.index.name])]
    head = " & ".join([str(n) for n in idx_names] + [str(c) for c in cols])
    lines.append(head.replace("_", "\\_").replace("%", "\\%") + " \\\\")
    lines.append("\\midrule")
    for idx, row in df.iterrows():
        idx = idx if isinstance(idx, tuple) else (idx,)
        cells = [str(i) for i in idx] + [floatfmt.format(v) if isinstance(v, (float, np.floating)) and np.isfinite(v)
                                         else ("--" if isinstance(v, (float, np.floating)) else str(v)) for v in row]
        lines.append(" & ".join(cells).replace("_", "\\_").replace("%", "\\%") + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

## 1. Data and audit
* **Campaign check.** The in-line 9-site and offline 89-site measurements are compared under all eight symmetries of the square grid (coordinates as given, rotations, reflections). The symmetry with the smallest residual is the one under which the two files describe the same wafer positions.
* **Map structure.** Wafer-to-wafer variation is split into a per-wafer offset and a shape; within-lot drift is measured per lot; the 9-site layout's own uniformity statistic is compared with the 89-site truth.

In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)
D = load_measurements(DATA_DIR, TARGET)
maps, meta, sites = D["maps"], D["meta"], D["sites"]
Y, lots, keys = maps.values, meta.lot.values, maps.index.values
xy, r = sites[["x", "y"]].values, sites.r.values
lab2i = {l: i for i, l in enumerate(sites.label)}
std_idx = {k: [lab2i[l] for l in v] for k, v in STANDARD_LAYOUTS.items()}
edge_idx = {k: [lab2i[l] for l in v] for k, v in EDGE_LAYOUTS.items()}
AUDIT = {}

# ---- gross measurement errors in the 89-point maps (replaced by the conditional mean of the wafer's other sites)
Y_raw = Y.copy()
Y, out_flags, out_z = clean_outliers(Y_raw, OUTLIER_Z)
maps = pd.DataFrame(Y, index=maps.index, columns=maps.columns)
outliers = [{"wafer": keys[w], "site": sites.label[j], "x_mm": sites.x[j], "y_mm": sites.y[j], "measured_um": round(Y_raw[w, j], 3),
             "replaced_by_um": round(Y[w, j], 3), "robust_z": round(out_z[w, j], 1)} for w, j in np.argwhere(out_flags)]
print(f"gross measurement errors (|robust z| > {OUTLIER_Z}): {len(outliers)}")
if outliers:
    display(pd.DataFrame(outliers))
AUDIT["outliers"] = outliers

# ---- what is in the data
per_lot = pd.DataFrame({"89-point wafers": meta.groupby("lot").size()})
if D["d9"] is not None:
    per_lot["9-point wafers"] = D["d9"].groupby("lot_number").experiment_key.nunique()
per_lot = D["lots"][["date_text", "type", "wafers_processed"]].join(per_lot)
print(f"{len(Y)} wafers with 89-point maps in {len(np.unique(lots))} lots; target = {TARGET}")
display(per_lot)
lot_date = meta.groupby("lot").date.first()
missing = [f"{lot_date[g]}_{w:02d}" for g, n in D["lots"]["wafers_processed"].items() if g in lot_date.index
           for w in range(1, int(n) + 1) if f"{lot_date[g]}_{w:02d}" not in set(keys)]
print("processed wafers without an 89-point map:", missing)
AUDIT.update(n_wafers=len(Y), n_lots=int(len(np.unique(lots))), wafers_per_lot=per_lot["89-point wafers"].to_dict(),
             missing_89pt=missing)

# ---- audit A: the two measurement campaigns (in-line 9-point vs offline 89-point)
if D["d9"] is not None:
    aud_si = audit_campaigns(D["d9"], D["d89"], "si_etch")
    aud_sh = audit_campaigns(D["d9"], D["d89"], "stepheight")
    print("\nAgreement of the campaigns under each grid symmetry - Si etch depth:")
    display(aud_si.round(3))
    print("... step height (the profilometer quantity itself):")
    display(aud_sh.round(3))
    best_tf = aud_sh["transform"].iloc[0]
    pairs_given = campaign_pairs(D["d9"], D["d89"], "si_etch", "identity")
    pairs_best = campaign_pairs(D["d9"], D["d89"], "si_etch", best_tf)
    site_off = (pairs_best.v9 - pairs_best.v89).groupby(pairs_best.loc_id).mean()
    b_sh = aud_sh.set_index("transform").loc[best_tf]
    AUDIT.update(campaign_best_transform=best_tf,
                 campaign_offset_stepheight_um=float(b_sh.mean_diff_um),
                 campaign_offset_si_etch_um=float(aud_si.set_index("transform").loc[best_tf].mean_diff_um),
                 campaign_resid_sd_stepheight_um=float(b_sh.resid_sd_after_wafer_offset_um),
                 campaign_resid_sd_identity_stepheight_um=float(aud_sh.set_index("transform").loc["identity"].resid_sd_after_wafer_offset_um),
                 campaign_runner_up_transform=str(aud_sh["transform"].iloc[1]),
                 campaign_resid_sd_runner_up_stepheight_um=float(aud_sh["resid_sd_after_wafer_offset_um"].iloc[1]),
                 campaign_resid_sd_after_site_offsets_stepheight_um=float(b_sh.resid_sd_after_site_offset_um),
                 noise_sd_per_tool_um=float(b_sh.resid_sd_after_site_offset_um / np.sqrt(2)),
                 campaign_within_wafer_corr_identity=float(aud_sh.set_index("transform").loc["identity"].median_within_wafer_corr),
                 campaign_within_wafer_corr_best=float(b_sh.median_within_wafer_corr))
    print(f"-> best symmetry: {best_tf} (step-height residual after a per-wafer offset {b_sh.resid_sd_after_wafer_offset_um:.3f} um "
          f"vs {AUDIT['campaign_resid_sd_identity_stepheight_um']:.3f} um as given and "
          f"{AUDIT['campaign_resid_sd_runner_up_stepheight_um']:.3f} um for the next best, {AUDIT['campaign_runner_up_transform']}); "
          f"offset {b_sh.mean_diff_um:+.3f} um; after per-site offsets the campaigns agree to "
          f"{b_sh.resid_sd_after_site_offset_um:.3f} um (noise per tool ~{AUDIT['noise_sd_per_tool_um']:.3f} um)")

# ---- audit B: how the maps vary
mu_map = Y.mean(0)
dev = Y - mu_map
shape = dev - dev.mean(1, keepdims=True)
_, sv, _ = np.linalg.svd(shape - shape.mean(0), full_matrices=False)
ev = sv ** 2 / (sv ** 2).sum()
lot_mean_shape = np.vstack([shape[lots == g].mean(0) for g in lots])
U = uniformity_metrics(Y, r)
drift = drift_table(maps, meta)
print(f"\nwithin-wafer s.d. {U['sd'].mean():.2f} um (CV {U['cv_pct'].mean():.2f} %), wafer-to-wafer s.d. of the mean "
      f"{U['mean'].std():.3f} um; edge minus centre {U['edge_minus_centre'].mean():.2f} um")
print(f"RMS deviation from the mean map {np.sqrt((dev ** 2).mean()):.3f} um; after removing each wafer's offset "
      f"{np.sqrt((shape ** 2).mean()):.3f} um; shape modes explain {np.round(np.cumsum(ev[:5]), 3)}")
print(f"share of shape variance that is lot-level: {(lot_mean_shape ** 2).mean() / (shape ** 2).mean():.2f}")
print("within-lot drift (wafer mean vs processing order):")
display(drift.round(3))
nine = Y[:, std_idx[9]]
naive_cv = 100 * nine.std(1) / nine.mean(1)
print(f"the 9-point layout's own CV is {naive_cv.mean():.2f} % vs {U['cv_pct'].mean():.2f} % on all 89 sites: it never "
      f"samples r > {r[std_idx[9]].max():.0f} mm, where the etch is {Y[:, r >= R_EDGE].mean() - Y[:, r < R_EDGE].mean():.1f} um deeper")
AUDIT.update(within_wafer_sd_um=float(U["sd"].mean()), cv_pct_true=float(U["cv_pct"].mean()),
             cv_pct_naive_9pt=float(naive_cv.mean()), wafer_mean_sd_um=float(U["mean"].std()),
             edge_minus_centre_um=float(U["edge_minus_centre"].mean()),
             rms_dev_from_mean_map_um=float(np.sqrt((dev ** 2).mean())),
             rms_shape_dev_um=float(np.sqrt((shape ** 2).mean())), shape_modes_cum_var=np.cumsum(ev[:5]).round(3).tolist(),
             lot_share_of_shape_var=float((lot_mean_shape ** 2).mean() / (shape ** 2).mean()),
             drift_corr_range=[float(drift.corr_mean_vs_order.min()), float(drift.corr_mean_vs_order.max())],
             drift_slope_um_per_wafer_mean=float(drift.slope_um_per_wafer.mean()))

learned_demo = [sites.label[i] for i in fsca_sites(Y, 5)]           # illustration only (all wafers)
fig_wafer(sites, Y, STANDARD_LAYOUTS[9], learned_demo, OUT_DIR)
if D["d9"] is not None:
    fig_audit(aud_sh, pairs_given, pairs_best, best_tf, OUT_DIR)
fig_drift(Y, meta, OUT_DIR)

## 2. Features: context, machine signals, OES
* **context**: wafer order in the lot and the lot's conditioning recipe (known before processing),
* **process**: per-phase (SF6 / C4F8) medians, spreads and trends over the ~100 Bosch cycles of every machine signal,
* **OES** (if the `Day_*.nc` files are present): single 40 ms spectra are noisy, so nothing is decided per spectrum. Per wafer: baseline = the dimmest 10 % of pixels; plasma-on = projection on the record-mean spectrum; the spectra are folded at the wafer's Bosch period (from the machine signals; from the spectra if missing) and the SF6 / C4F8 window is the one that best separates them; features = emission share of 10 nm bands, continuum-subtracted line intensities, log total emission and early-to-late trends, over the whole etch and per phase. The printed checks should show plasma-on ≈ 1, SF6 share ≈ 0.72 and a phase contrast well above 1.

Feature extraction uses no measurement, so it cannot leak labels; scaling and penalties are still fitted inside the training lots.

In [ ]:
C = context_features(meta)
decoded = find_file(DATA_DIR, FILES["proc_dict"], required=False) is not None
proc_cache = os.path.join(OUT_DIR, f"features_process_v2_{'decoded' if decoded else 'codes'}.csv")
if os.path.exists(proc_cache):
    Fp = pd.read_csv(proc_cache, index_col=0)
    print(f"process features loaded from {proc_cache}: {Fp.shape}")
else:
    t0 = time.time()
    Fp, proc_info = process_features(DATA_DIR, keys=set(keys))
    Fp.to_csv(proc_cache)
    pd.DataFrame.from_dict(proc_info, orient="index").to_csv(proc_cache.replace(".csv", "_diagnostics.csv"))
    print(f"  ({time.time() - t0:.0f} s)")
Fp = Fp.reindex(maps.index)
_pd_path = proc_cache.replace(".csv", "_diagnostics.csv")
if os.path.exists(_pd_path):                     # did the Bosch phase detection behave? (expected: 100 cycles, 4.5 s / 1.5 s)
    _pdg = pd.read_csv(_pd_path, index_col=0)
    _ok = _pdg[_pdg["ok"].astype(str) == "True"]
    AUDIT["process_checks"] = {
        "wafers": int(len(_pdg)), "phases_found": int(len(_ok)),
        "sf6_channel": _ok["sf6_channel"].mode().tolist()[:1] if len(_ok) else None,
        "c4f8_channel": _ok["c4f8_channel"].mode().tolist()[:1] if len(_ok) else None,
        "n_cycles_median": float(_ok["n_cycles"].median()) if "n_cycles" in _ok else None,
        "sf6_pulse_s_median": float(_ok["sf6_pulse_s"].median()) if "sf6_pulse_s" in _ok else None,
        "c4f8_pulse_s_median": float(_ok["c4f8_pulse_s"].median()) if "c4f8_pulse_s" in _ok else None,
        "cycle_period_s_median": float(_ok["cycle_period_s"].median()) if "cycle_period_s" in _ok else None,
        "sf6_duty_median": float(_ok["sf6_duty"].median()) if "sf6_duty" in _ok else None}
    print("process-data checks:", AUDIT["process_checks"])
    # Bosch cycle timing of every wafer (period, SF6 duty, first SF6 start) - used to fold that wafer's OES
    TIMING = {k: {"period": r_.cycle_period_s, "duty": r_.sf6_duty, "t0": r_.t0_sf6_epoch_s}
              for k, r_ in _ok.iterrows() if "cycle_period_s" in _ok and np.isfinite(r_.cycle_period_s)}
else:
    TIMING = {}
print("wafers without process data:", list(Fp.index[Fp.isna().all(1)]))

Fo = None
if USE_OES:
    oes_cache = os.path.join(OUT_DIR, "features_oes_v2.csv")
    Fo = oes_features(DATA_DIR, keys=set(keys), cache=oes_cache, timing=TIMING or None,
                      debug_path=os.path.join(OUT_DIR, "oes_debug.npz"))
    _od_path = oes_cache.replace(".csv", "_diagnostics.csv")
    if os.path.exists(_od_path):                 # did plasma-on and SF6/C4F8 phase detection behave? (expected SF6 ~0.75)
        _odg = pd.read_csv(_od_path, index_col=0)

        def _q(c, fn=np.nanmedian):
            return float(fn(_odg[c])) if c in _odg and _odg[c].notna().any() else None
        AUDIT["oes_checks"] = {
            "wafers": int(len(_odg)), "on_fraction_median": _q("diag|on_fraction"),
            "sf6_fraction_median": _q("diag|sf6_fraction"), "sf6_fraction_min": _q("diag|sf6_fraction", np.nanmin),
            "sf6_fraction_max": _q("diag|sf6_fraction", np.nanmax), "phase_contrast_median": _q("diag|phase_contrast"),
            "phase_contrast_min": _q("diag|phase_contrast", np.nanmin), "period_s_median": _q("diag|period_s"),
            "F703_line_snr_median": _q("diag|line_snr_F703"), "sampling_interval_s_median": _q("diag|dt_median_s"),
            "phase_method": _odg["diag|phase_method"].astype(str).value_counts().to_dict() if "diag|phase_method" in _odg else None,
            "code_dtype": _odg["diag|code_dtype"].astype(str).value_counts().to_dict() if "diag|code_dtype" in _odg else None,
            "code_range": [_q("diag|code_min", np.nanmin), _q("diag|code_max", np.nanmax)],
            "codes_beyond_dictionary": int(_odg["diag|codes_beyond_dictionary"].fillna(0).sum()) if "diag|codes_beyond_dictionary" in _odg else 0}
    if Fo is not None and len(Fo.columns):
        Fo = Fo.reindex(maps.index)
        cover = 1 - Fo.isna().all(1).mean()
        print(f"OES available for {cover:.0%} of the wafers; without OES:", list(Fo.index[Fo.isna().all(1)]))
        if cover < OES_MIN_COVERAGE:
            print(f"-> below OES_MIN_COVERAGE = {OES_MIN_COVERAGE:.0%}: OES levels skipped (are all Day_*.nc files in DATA_DIR?)")
            Fo = None
    else:
        Fo = None

levels = {"none": None, "context": C.values, "process": Fp.values, "context+process": np.c_[C.values, Fp.values]}
if Fo is not None:
    levels["OES"] = Fo.values
    levels["context+process+OES"] = np.c_[C.values, Fp.values, Fo.values]
LEVELS_ORDER = list(levels)
BEST = LEVELS_ORDER[-1]          # richest information level
print("information levels:", {k: (0 if v is None else v.shape[1]) for k, v in levels.items()}, "-> richest:", BEST)

# which signals carry the wafer-to-wafer offset? (ridge on all wafers, standardized coefficients; descriptive only)
Xall = levels[BEST]
names = list(C.columns) + list(Fp.columns) + ([] if Fo is None else list(Fo.columns))
vm_all = MapVM(N_MODES).fit(Y, Xall, lots)
coef = pd.Series(vm_all.W[:, 0], index=names)
top = coef.abs().sort_values(ascending=False).head(12).index
Xf = pd.DataFrame(np.where(np.isnan(Xall), nan_fill(Xall), Xall), index=maps.index, columns=names)
expl = pd.DataFrame({"coef_offset_model": coef[top],
                     "corr_with_wafer_mean": [np.corrcoef(Xf[c], Y.mean(1))[0, 1] for c in top],
                     "corr_with_wafer_order": [np.corrcoef(Xf[c], meta.wafer)[0, 1] for c in top]})
print("signals with the largest weight in the offset model (all wafers, descriptive):")
display(expl.round(3))
expl.to_csv(os.path.join(OUT_DIR, "tables", "offset_model_top_signals.csv"))

## 3. Experiments (leave-one-lot-out)
The reconstruction is Gaussian conditioning: prior mean = virtual-metrology prediction of the whole map (training mean map when no features), prior covariance = covariance of out-of-fold residuals on the training lots + a nugget, measurement noise `NOISE_SD`. Site designs are learned on the training lots: **FSCA** (McLoone, Johnston & Susto, IEEE TASE 2018), **QR pivoting** (Manohar et al., 2018), **greedy Bayesian** (largest reduction of the summed posterior variance), against the **standard** 1/5/9-site layouts, the **edge-inclusive** 5/9-site layouts and **random** sites. Baselines on the same sites: inverse-distance weighting (no history; used by the dataset authors to fill gaps) and, using the training lots, mean map + offset, mean map + IDW of the residuals, mean map + Zernike fit and stationary exponential kriging. The wafer offset of the prior is an elastic net (`OFFSET_MODEL`), the shape scores ridge regressions. A forward-in-time run predicts each lot from earlier lots only.

In [ ]:
COLS = ["rmse", "rmse_inner", "rmse_edge", "max_abs"] + [f"err_{m}" for m in METRICS]
RAW = os.path.join(OUT_DIR, "raw")
t_all = time.time()

# ---- RQ1: where to measure, without process data
t0 = time.time()
res_sites = exp_site_selection(Y, lots, keys, xy, r, std_idx, K_MAX, NOISE_VAR, NUGGET, N_RANDOM, SEED, edge_idx)
res_sites.to_csv(os.path.join(RAW, "rq1_site_selection.csv"), index=False)
m_sites = macro(res_sites, ["design", "estimator", "k"], COLS, N_BOOT, SEED)
print(f"RQ1 done ({time.time() - t0:.0f} s)")
display(m_sites[m_sites.k.isin([0, 1, 3, 5, 9]) & (m_sites.estimator == "Kriging")]
        .pivot_table(index="design", columns="k", values="rmse").round(3))
print("same sites, different estimators (k = 9; IDW uses no history, the others use the training lots):")
display(m_sites[(m_sites.k == 9) & m_sites.design.isin(["Standard", "Edge-inclusive", "FSCA"])]
        .pivot_table(index="estimator", columns="design", values="rmse").round(3))

# ---- RQ2: what process information is worth
t0 = time.time()
res_info, ridge_alphas = exp_information(Y, lots, keys, r, levels, std_idx, K_MAX, NOISE_VAR, NUGGET, N_MODES)
res_info.to_csv(os.path.join(RAW, "rq2_information.csv"), index=False)
m_info = macro(res_info, ["level", "design", "k"], COLS, N_BOOT, SEED)
print(f"RQ2 done ({time.time() - t0:.0f} s)")
print("same measured sites for every level (FSCA design) - isolates what the process prior adds:")
display(m_info[m_info.k.isin([0, 1, 2, 3, 5, 9]) & (m_info.design == "FSCA")]
        .pivot_table(index="level", columns="k", values="rmse").loc[LEVELS_ORDER].round(3))
print("each level picks its own sites (greedy under its residual covariance):")
display(m_info[m_info.k.isin([0, 1, 2, 3, 5, 9]) & (m_info.design == "Greedy")]
        .pivot_table(index="level", columns="k", values="rmse").loc[LEVELS_ORDER].round(3))
equiv = []
for dname in ("FSCA", "Greedy"):
    ref = m_info[(m_info.level == "none") & (m_info.design == dname)].set_index("k").rmse.sort_index()
    for lname in LEVELS_ORDER:
        cur = m_info[(m_info.level == lname) & (m_info.design == dname)].set_index("k").rmse.sort_index()
        for k in (0, 1, 2, 3, 5):
            e = sites_to_match(ref, cur.loc[k])
            equiv.append({"design": dname, "level": lname, "k": k, "rmse": cur.loc[k],
                          "sites_needed_without_process_data": e, "sites_saved": e - k})
equiv = pd.DataFrame(equiv)
print("site-equivalent value of the information (extra sites the measurement-only model needs for the same RMSE):")
display(equiv.pivot_table(index=["design", "level"], columns="k", values="sites_saved").round(2))

# ---- RQ3: can process information choose the sites per wafer?
t0 = time.time()
feat_levels = {k: v for k, v in levels.items() if v is not None and k in ("process", BEST)}
res_adapt = exp_adaptive(Y, lots, keys, r, feat_levels, min(K_MAX, 9), NOISE_VAR, NUGGET, N_MODES)
res_adapt.to_csv(os.path.join(RAW, "rq3_adaptive.csv"), index=False)
m_adapt = macro(res_adapt, ["level", "design", "k"], COLS, N_BOOT, SEED)
print(f"RQ3 done ({time.time() - t0:.0f} s)")
display(m_adapt.pivot_table(index=["level", "design"], columns="k", values="rmse").round(3))

# ---- RQ4: are the uncertainties trustworthy on unseen lots?
t0 = time.time()
cal_levels = {k: levels[k] for k in dict.fromkeys(["none", "context", BEST])}
res_cal = exp_calibration(Y, lots, keys, r, cal_levels, (0, 1, 2, 3, 5, 9), NOISE_VAR, NUGGET, ALPHA, N_MODES)
res_cal.to_csv(os.path.join(RAW, "rq4_calibration.csv"), index=False)
cal_cols = ["cov_site_gauss", "cov_site_conformal", "halfwidth_site_gauss", "halfwidth_site_conformal", "mean_z2",
            "cov_mean", "halfwidth_mean", "cov_cv_pct", "halfwidth_cv_pct", "cov_sd_inner", "halfwidth_sd_inner"]
m_cal = macro(res_cal, ["level", "k"], cal_cols, N_BOOT, SEED)
print(f"RQ4 done ({time.time() - t0:.0f} s)")
display(m_cal[["level", "k", "cov_site_gauss", "cov_site_conformal", "halfwidth_site_conformal", "cov_cv_pct",
               "halfwidth_cv_pct", "cov_mean", "halfwidth_mean"]].round(3))

# ---- RQ5: can process information decide which wafers need no measurement?
t0 = time.time()
res_skip = exp_measure_or_skip(Y, lots, keys, r, cal_levels, TAUS, NOISE_VAR, NUGGET, ALPHA, N_MODES)
res_skip.to_csv(os.path.join(RAW, "rq5_measure_or_skip.csv"), index=False)
m_skip = macro(res_skip, ["level", "tau"], ["skipped", "within_tau", "err_mean", "rmse"], N_BOOT, SEED)
_sk = res_skip[res_skip.skipped == 1].groupby(["level", "tau"])
m_skip = m_skip.merge(pd.DataFrame({"n_skipped": _sk.size(), "skipped_within_tau": _sk.within_tau.mean()}).reset_index(),
                      on=["level", "tau"], how="left")
m_skip["n_skipped"] = m_skip["n_skipped"].fillna(0).astype(int)
print(f"RQ5 done ({time.time() - t0:.0f} s)")
display(m_skip[["level", "tau", "skipped", "n_skipped", "skipped_within_tau", "within_tau", "err_mean", "rmse"]].round(3))
print(f"(a skipped wafer should be within tau at least {1 - ALPHA:.0%} of the time - the guarantee holds only if lots are exchangeable)")

# ---- robustness: forward in time (train only on earlier lots)
t0 = time.time()
fwd_levels = {k: levels[k] for k in dict.fromkeys(["none", "context", BEST])}
res_fwd = exp_forward(Y, lots, keys, r, std_idx, edge_idx, fwd_levels, NOISE_VAR, NUGGET, N_MODES)
res_fwd.to_csv(os.path.join(RAW, "forward_in_time.csv"), index=False)
m_fwd = macro(res_fwd, ["level", "design", "k"], COLS, N_BOOT, SEED)
print(f"forward-in-time check done ({time.time() - t0:.0f} s); test lots {sorted(res_fwd.lot.unique())}")
display(m_fwd.pivot_table(index=["design", "k"], columns="level", values="rmse").round(3))

# ---- RQ6: the real in-line 9-point measurements as the sparse input
if D["d9"] is not None:
    t0 = time.time()
    res_inline, inline_tf = exp_inline(D, Y, lots, keys, r, {k: levels[k] for k in dict.fromkeys(["none", BEST])},
                                       NOISE_VAR, NUGGET, TARGET, N_MODES)
    res_inline.to_csv(os.path.join(RAW, "rq6_inline.csv"), index=False)
    m_inline = macro(res_inline, ["level", "case"], COLS, N_BOOT, SEED)
    print(f"RQ6 done ({time.time() - t0:.0f} s); orientation learned per fold:", inline_tf["transform"].value_counts().to_dict())
    display(m_inline[["level", "case", "rmse", "rmse_inner", "rmse_edge", "err_mean", "err_cv_pct"]].round(3))

# ---- paired comparisons behind the paper's claims (per-lot differences, lot bootstrap)
def _sel(df, **kw):
    for k, v in kw.items():
        df = df[df[k] == v]
    return df


PAIRS = {
    "3 learned (FSCA) vs standard 9": (_sel(res_sites, design="FSCA", estimator="Kriging", k=3),
                                       _sel(res_sites, design="Standard", estimator="Kriging", k=9)),
    "9 learned (greedy) vs standard 9": (_sel(res_sites, design="Greedy", estimator="Kriging", k=9),
                                         _sel(res_sites, design="Standard", estimator="Kriging", k=9)),
    "1 learned (FSCA) vs centre site": (_sel(res_sites, design="FSCA", estimator="Kriging", k=1),
                                        _sel(res_sites, design="Standard", estimator="Kriging", k=1)),
    "standard 9: learned model vs IDW": (_sel(res_sites, design="Standard", estimator="Kriging", k=9),
                                         _sel(res_sites, design="Standard", estimator="IDW", k=9)),
    f"0 sites: {BEST} vs none": (_sel(res_info, design="FSCA", level=BEST, k=0), _sel(res_info, design="FSCA", level="none", k=0)),
    f"0 sites: {BEST} vs context": (_sel(res_info, design="FSCA", level=BEST, k=0), _sel(res_info, design="FSCA", level="context", k=0)),
    f"1 site (FSCA): {BEST} vs none": (_sel(res_info, design="FSCA", level=BEST, k=1), _sel(res_info, design="FSCA", level="none", k=1)),
    f"3 sites (FSCA): {BEST} vs none": (_sel(res_info, design="FSCA", level=BEST, k=3), _sel(res_info, design="FSCA", level="none", k=3)),
    f"3 sites: adaptive vs static ({BEST})": (_sel(res_adapt, level=BEST, design="Adaptive", k=3),
                                              _sel(res_adapt, level=BEST, design="Static", k=3)),
    "4 learned (FSCA) vs standard 9": (_sel(res_sites, design="FSCA", estimator="Kriging", k=4),
                                       _sel(res_sites, design="Standard", estimator="Kriging", k=9)),
    "edge-inclusive 9 vs standard 9": (_sel(res_sites, design="Edge-inclusive", estimator="Kriging", k=9),
                                       _sel(res_sites, design="Standard", estimator="Kriging", k=9)),
    "9 learned (FSCA) vs edge-inclusive 9": (_sel(res_sites, design="FSCA", estimator="Kriging", k=9),
                                             _sel(res_sites, design="Edge-inclusive", estimator="Kriging", k=9)),
    "standard 9: learned model vs mean map + offset": (_sel(res_sites, design="Standard", estimator="Kriging", k=9),
                                                       _sel(res_sites, design="Standard", estimator="Offset", k=9)),
    "standard 9: learned model vs mean map + IDW residual": (_sel(res_sites, design="Standard", estimator="Kriging", k=9),
                                                             _sel(res_sites, design="Standard", estimator="IDW-residual", k=9)),
    "standard 9: learned model vs stationary kriging": (_sel(res_sites, design="Standard", estimator="Kriging", k=9),
                                                        _sel(res_sites, design="Standard", estimator="Stationary", k=9)),
    f"3 sites (FSCA) + {BEST} vs standard 9 (no process data)": (_sel(res_info, design="FSCA", level=BEST, k=3),
                                                                  _sel(res_sites, design="Standard", estimator="Kriging", k=9)),
    "forward: 9 learned (FSCA) vs standard 9": (_sel(res_fwd, level="none", design="FSCA", k=9),
                                                _sel(res_fwd, level="none", design="Standard", k=9)),
    f"forward: 1 site (FSCA), {BEST} vs none": (_sel(res_fwd, level=BEST, design="FSCA", k=1),
                                                _sel(res_fwd, level="none", design="FSCA", k=1)),
    f"forward: 3 sites (FSCA), {BEST} vs none": (_sel(res_fwd, level=BEST, design="FSCA", k=3),
                                                 _sel(res_fwd, level="none", design="FSCA", k=3)),
}
if "context+process" in levels and BEST != "context+process":       # does OES add to the machine signals?
    for k in (0, 1, 3):
        PAIRS[f"{k} sites (FSCA): {BEST} vs context+process"] = (_sel(res_info, design="FSCA", level=BEST, k=k),
                                                                 _sel(res_info, design="FSCA", level="context+process", k=k))
if D["d9"] is not None:
    al = [c for c in res_inline.case.unique() if c.startswith("in-line") and c.endswith("+ site offsets") and "as given" not in c][0]
    PAIRS["in-line 9 aligned vs offline 9"] = (_sel(res_inline, level="none", case=al),
                                               _sel(res_inline, level="none", case="offline values at the same sites"))
    PAIRS["in-line 9: as given vs aligned"] = (_sel(res_inline, level="none", case="in-line, coordinates as given + site offsets"),
                                               _sel(res_inline, level="none", case=al))
paired_tab = pd.DataFrame({name: paired(a, b, "rmse", N_BOOT, SEED) for name, (a, b) in PAIRS.items()}).T
paired_tab.to_csv(os.path.join(OUT_DIR, "tables", "paired_comparisons_rmse.csv"))
print("paired comparisons of map RMSE (um; negative = first method better):")
display(paired_tab.round(4))

# ---- uniformity metric estimates for the figure
unif = uniformity_estimates(Y, lots, r, std_idx, levels[BEST], NOISE_VAR, NUGGET, "cv_pct", N_MODES)
unif.to_csv(os.path.join(RAW, "uniformity_cv_errors.csv"), index=False)
print(f"all experiments: {(time.time() - t_all) / 60:.1f} min")

## 4. Tables and figures for the paper

In [ ]:
TAB = os.path.join(OUT_DIR, "tables")


def ci(df, col, fmt="{:.3f}"):
    """'value [lo, hi]' strings for a macro table column."""
    return [f"{fmt.format(v)} [{fmt.format(lo)}, {fmt.format(hi)}]" for v, lo, hi in zip(df[col], df[col + "_lo"], df[col + "_hi"])]


# ---- Table: site selection (no process data), learned-covariance estimator
t = m_sites[m_sites.k.isin([1, 3, 4, 5, 9]) & (m_sites.estimator == "Kriging")].copy()
t["RMSE (um) [95% CI]"] = ci(t, "rmse")
t = t.rename(columns={"rmse_inner": "RMSE inner", "rmse_edge": "RMSE edge", "err_cv_pct": "|CV error| (pp)"})
tab_sites = t.set_index(["design", "k"])[["RMSE (um) [95% CI]", "RMSE inner", "RMSE edge", "|CV error| (pp)"]]
tab_sites.to_csv(os.path.join(TAB, "table_rq1_site_selection.csv"))
latex_table(tab_sites.round(3), os.path.join(TAB, "table_rq1_site_selection.tex"),
            "Map reconstruction on unseen lots (leave-one-lot-out, mean over lots) without process data.", "tab:sites")
display(tab_sites)

# ---- Table: estimators on the same sites (is the learned covariance needed, or is the mean map enough?)
t = m_sites[m_sites.k.isin([3, 9]) & m_sites.design.isin(["FSCA", "Standard", "Edge-inclusive"])].copy()
t["cell"] = ci(t, "rmse")
tab_est = t.pivot_table(index="estimator", columns=["design", "k"], values="cell", aggfunc="first")
tab_est = tab_est.reindex(["Kriging", "Stationary", "IDW-residual", "Offset", "Zernike", "IDW"])
tab_est.columns = [f"{d} {k}" for d, k in tab_est.columns]
tab_est.index = tab_est.index.map({"Kriging": "learned covariance (ours)", "Stationary": "stationary kriging",
                                   "IDW-residual": "mean map + IDW of residuals", "Offset": "mean map + offset",
                                   "Zernike": "mean map + Zernike", "IDW": "IDW (no history)"})
tab_est.to_csv(os.path.join(TAB, "table_rq1_estimators.csv"))
latex_table(tab_est, os.path.join(TAB, "table_rq1_estimators.tex"),
            "Map RMSE (um) of estimators on the same sites. Kriging = learned covariance (ours); Stationary = exponential "
            "covariance; IDW-residual = mean map + IDW of the deviations; Offset = mean map + offset; IDW without history.",
            "tab:estimators")
display(tab_est)

# ---- Table: information levels
t = m_info[(m_info.design == "FSCA") & m_info.k.isin([0, 1, 3, 5])].copy()
t["cell"] = ci(t, "rmse")
tab_info = t.pivot_table(index="level", columns="k", values="cell", aggfunc="first").loc[LEVELS_ORDER]
tab_info.columns = [f"k={c}" for c in tab_info.columns]
eq = equiv[equiv.design == "FSCA"]
tab_info["sites saved at k=0"] = eq[eq.k == 0].set_index("level").sites_saved.loc[LEVELS_ORDER].round(2)
tab_info["sites saved at k=1"] = eq[eq.k == 1].set_index("level").sites_saved.loc[LEVELS_ORDER].round(2)
tab_info.to_csv(os.path.join(TAB, "table_rq2_information.csv"))
latex_table(tab_info, os.path.join(TAB, "table_rq2_information.tex"),
            "Map RMSE (um, 95\\% lot-bootstrap CI) by information level, with the same k measured sites (FSCA) for every level.", "tab:info")
display(tab_info)

# ---- Table: calibration
tab_cal = m_cal[m_cal.k.isin([0, 1, 3, 9])].set_index(["level", "k"])[
    ["cov_site_gauss", "cov_site_conformal", "halfwidth_site_conformal", "cov_mean", "halfwidth_mean", "cov_cv_pct",
     "halfwidth_cv_pct"]]
tab_cal.to_csv(os.path.join(TAB, "table_rq4_calibration.csv"))
latex_table(tab_cal, os.path.join(TAB, "table_rq4_calibration.tex"),
            "Coverage of 90\\% intervals on unseen lots and their half-widths (um or percentage points).", "tab:cal")

# ---- Table: measure or skip
tab_skip = m_skip.set_index(["level", "tau"])[["skipped", "n_skipped", "skipped_within_tau", "within_tau", "err_mean", "rmse"]]
tab_skip.to_csv(os.path.join(TAB, "table_rq5_measure_or_skip.csv"))
latex_table(tab_skip, os.path.join(TAB, "table_rq5_measure_or_skip.tex"),
            "Wafers allowed to skip metrology, share of the skipped wafers within the tolerance (target >= 0.9) and of all "
            "wafers.", "tab:skip")

# ---- Table: in-line campaign
if D["d9"] is not None:
    tab_inline = m_inline.set_index(["level", "case"])[["rmse", "rmse_inner", "rmse_edge", "err_mean", "err_cv_pct"]]
    tab_inline.to_csv(os.path.join(TAB, "table_rq6_inline.csv"))
    latex_table(tab_inline, os.path.join(TAB, "table_rq6_inline.tex"),
                "Reconstructing the offline 89-point map from the in-line 9-point measurements.", "tab:inline")

# ---- Table: forward in time
tab_fwd = m_fwd.pivot_table(index=["design", "k"], columns="level", values="rmse")
tab_fwd.to_csv(os.path.join(TAB, "table_forward_in_time.csv"))
latex_table(tab_fwd.round(3), os.path.join(TAB, "table_forward_in_time.tex"),
            "Map RMSE (um) when each lot is predicted from earlier lots only.", "tab:forward")

# ---- Table: adaptive vs static designs
tab_adapt = m_adapt.pivot_table(index=["level", "design"], columns="k", values="rmse")
tab_adapt.to_csv(os.path.join(TAB, "table_rq3_adaptive.csv"))

# ---- figures
fig_curves(m_sites, m_info, LEVELS_ORDER, OUT_DIR, "rmse", "map RMSE (um)", design_b="FSCA")
fig_curves(m_sites, m_info, LEVELS_ORDER, OUT_DIR, "err_cv_pct", "|CV error| (percentage points)", design_b="FSCA")
fig_uniformity(unif.drop(columns="lot"), OUT_DIR, "cv_pct", "CV (pp)")
fig_calibration(m_cal, list(cal_levels), OUT_DIR, 1 - ALPHA)

## 5. Sensitivity checks (target, nugget, number of shape modes)

In [ ]:
sens = []
if RUN_SENSITIVITY:
    t0 = time.time()
    lv = {k: levels[k] for k in dict.fromkeys(["none", BEST])}

    def _run(Ymat, tag, nugget=NUGGET, n_modes=N_MODES):
        res, _ = exp_information(Ymat, lots, keys, r, lv, std_idx, 9, NOISE_VAR, nugget, n_modes)
        m = macro(res[res.design == "FSCA"], ["level", "k"], ["rmse", "err_cv_pct"], 200, SEED)
        m["setting"] = tag
        return m

    sens.append(_run(Y, f"main: {TARGET}, nugget {NUGGET:.0e}, {N_MODES} modes"))
    other = "stepheight" if TARGET != "stepheight" else "si_etch"
    sens.append(_run(D["extra"][other].loc[maps.index].values, f"target = {other}"))
    sens.append(_run(Y_raw, "raw maps (gross errors kept)"))
    sens.append(_run(Y, "nugget x 10", nugget=10 * NUGGET))
    sens.append(_run(Y, "2 shape modes", n_modes=2))
    sens.append(_run(Y, "8 shape modes", n_modes=8))
    _om = OFFSET_MODEL                                   # the ridge offset model of v1, for comparison
    OFFSET_MODEL = "ridge"
    sens.append(_run(Y, "ridge offset model (v1 of this notebook)"))
    OFFSET_MODEL = _om
    sens = pd.concat(sens, ignore_index=True)
    tab_sens = sens[sens.k.isin([0, 1, 3, 5, 9])].pivot_table(index=["setting", "level"], columns="k", values="rmse")
    tab_sens.to_csv(os.path.join(TAB, "table_sensitivity.csv"))
    print(f"sensitivity done ({time.time() - t0:.0f} s)")
    display(tab_sens.round(3))

## 6. Summary (`results.json`) - send this output back

In [ ]:
def _get(m, **kw):
    d = m
    for k, v in kw.items():
        d = d[d[k] == v]
    return d


def _val(m, col="rmse", **kw):
    d = _get(m, **kw)
    return float(d[col].iloc[0]) if len(d) else float("nan")


S = {"audit": AUDIT, "settings": {"target": TARGET, "k_max": K_MAX, "noise_sd_um": NOISE_SD, "n_modes": N_MODES,
                                  "alpha": ALPHA, "outlier_z": OUTLIER_Z,
                                  "levels": {k: (0 if v is None else int(v.shape[1])) for k, v in levels.items()},
                                  "process_features_decoded": bool(decoded), "oes_used": Fo is not None}}
kr = m_sites[m_sites.estimator == "Kriging"]
std9 = _val(kr, design="Standard", k=9)
S["rq1_where_to_measure"] = {
    "rmse_um": {d: {int(k): round(_val(kr, design=d, k=k), 4) for k in (1, 3, 5, 9)} for d in ("FSCA", "Greedy", "QR", "Random")},
    "standard_kriging_rmse_um": {int(k): round(_val(kr, design="Standard", k=k), 4) for k in (1, 5, 9)},
    "standard9_idw_rmse_um": round(_val(m_sites, design="Standard", estimator="IDW", k=9), 4),
    "standard9_zernike_rmse_um": round(_val(m_sites, design="Standard", estimator="Zernike", k=9), 4),
    "edge_inclusive_kriging_rmse_um": {int(k): round(_val(kr, design="Edge-inclusive", k=k), 4) for k in (5, 9)},
    "estimators_rmse_um": {f"{d} {k}": {e: round(_val(m_sites, design=d, estimator=e, k=k), 4)
                                        for e in ("Kriging", "Stationary", "IDW-residual", "Offset", "Zernike", "IDW")}
                           for d in ("Standard", "Edge-inclusive", "FSCA") for k in (3, 9)
                           if len(_get(m_sites, design=d, k=k))},
    "sites_to_match_standard9": {d: round(sites_to_match(kr[kr.design == d].set_index("k").rmse.sort_index(), std9), 2)
                                 for d in ("FSCA", "Greedy", "QR", "Random")},
    "cv_error_pp_mean_signed": {c.replace("\n", " "): round(float(unif[c].mean()), 3) for c in unif.columns if c != "lot"},
    "cv_error_pp_mean_abs": {c.replace("\n", " "): round(float(unif[c].abs().mean()), 3) for c in unif.columns if c != "lot"},
}
S["rq2_value_of_information"] = {
    dname: {"rmse_um": {l: {int(k): round(_val(m_info, level=l, design=dname, k=k), 4) for k in (0, 1, 3, 5, 9)} for l in LEVELS_ORDER},
            "sites_saved": {l: {int(k): round(float(_val(equiv, "sites_saved", design=dname, level=l, k=k)), 2) for k in (0, 1, 3)}
                            for l in LEVELS_ORDER}}
    for dname in ("FSCA", "Greedy")}
S["rq3_adaptive_vs_static_rmse_um"] = {f"{l} / {d}": {int(k): round(_val(m_adapt, level=l, design=d, k=k), 4) for k in (1, 3, 5)}
                                       for l in feat_levels for d in ("Static", "Adaptive")}
S["rq4_calibration"] = {l: {int(k): {c: round(_val(m_cal, c, level=l, k=k), 3)
                                     for c in ("cov_site_gauss", "cov_site_conformal", "halfwidth_site_conformal",
                                               "cov_mean", "halfwidth_mean", "cov_cv_pct", "halfwidth_cv_pct")}
                            for k in (0, 1, 3)} for l in cal_levels}
S["rq5_measure_or_skip"] = {l: {float(t): {c: round(_val(m_skip, c, level=l, tau=t), 3)
                                           for c in ("skipped", "n_skipped", "skipped_within_tau", "within_tau", "rmse")}
                                for t in TAUS} for l in cal_levels}
S["forward_in_time_rmse_um"] = {f"{l} / {d} {k}": round(_val(m_fwd, level=l, design=d, k=k), 4)
                                for l, d, k in m_fwd[["level", "design", "k"]].drop_duplicates().itertuples(index=False)}
if D["d9"] is not None:
    S["rq6_inline"] = {f"{l} / {c}": {col: round(_val(m_inline, col, level=l, case=c), 4) for col in ("rmse", "err_mean", "err_cv_pct")}
                       for l, c in m_inline[["level", "case"]].drop_duplicates().itertuples(index=False)}
S["paired_rmse_comparisons"] = {k: {c: (round(v, 4) if isinstance(v, float) else v) for c, v in row.items()}
                                for k, row in paired_tab.to_dict(orient="index").items()}
if RUN_SENSITIVITY and len(sens):
    S["sensitivity_rmse_um"] = {f"{s} / {l}": {int(k): round(_val(sens, setting=s, level=l, k=k), 4) for k in (0, 1, 3, 9)}
                                for s, l in sens[["setting", "level"]].drop_duplicates().itertuples(index=False)}

# ---- the headline numbers in words
A, R1, R2 = AUDIT, S["rq1_where_to_measure"], S["rq2_value_of_information"]["FSCA"]
P = S["paired_rmse_comparisons"]
lines = [
    f"DATA  {A['n_wafers']} wafers, {A['n_lots']} lots; {len(A.get('outliers', []))} gross measurement errors replaced; "
    f"within-wafer s.d. {A['within_wafer_sd_um']:.2f} um (CV {A['cv_pct_true']:.2f} %), wafer-mean s.d. {A['wafer_mean_sd_um']:.2f} um, "
    f"within-lot drift r = {A['drift_corr_range'][0]:.2f} to {A['drift_corr_range'][1]:.2f}.",
]
if "campaign_best_transform" in A:
    lines.append(f"AUDIT the 9-point and 89-point step heights match after {A['campaign_best_transform']}: residual after a per-wafer "
                 f"offset {A['campaign_resid_sd_stepheight_um']:.3f} um vs {A['campaign_resid_sd_identity_stepheight_um']:.2f} um as given "
                 f"and {A['campaign_resid_sd_runner_up_stepheight_um']:.3f} um for the next best ({A['campaign_runner_up_transform']}); "
                 f"tool offset {A['campaign_offset_stepheight_um']:+.2f} um.")
if "process_checks" in A:
    pc = A["process_checks"]
    lines.append(f"CHECK process data: Bosch phases found on {pc['phases_found']}/{pc['wafers']} wafers, cycle period "
                 f"{pc.get('cycle_period_s_median') or float('nan'):.3f} s, SF6 duty {pc.get('sf6_duty_median') or float('nan'):.2f}.")
if "oes_checks" in A:
    oc = A["oes_checks"]
    lines.append(f"CHECK OES: {oc['wafers']} wafers, plasma-on {oc.get('on_fraction_median') or float('nan'):.2f}, SF6-phase share "
                 f"{oc.get('sf6_fraction_median') or float('nan'):.2f}, phase contrast {oc.get('phase_contrast_median') or float('nan'):.1f} "
                 f"(min {oc.get('phase_contrast_min') or float('nan'):.1f}), methods {oc.get('phase_method')}.")
lines += [
    f"RQ1   the 9-point layout's own CV is {A['cv_pct_naive_9pt']:.2f} % vs {A['cv_pct_true']:.2f} % on 89 sites; "
    f"standard 9 + learned model {R1['standard_kriging_rmse_um'][9]:.3f} um (mean map + offset "
    f"{R1['estimators_rmse_um'].get('Standard 9', {}).get('Offset', float('nan')):.3f}, stationary kriging "
    f"{R1['estimators_rmse_um'].get('Standard 9', {}).get('Stationary', float('nan')):.3f}, IDW {R1['standard9_idw_rmse_um']:.2f}); "
    f"edge-inclusive 9 {R1['edge_inclusive_kriging_rmse_um'].get(9, float('nan')):.3f} um; learned sites match the standard 9 with "
    f"{R1['sites_to_match_standard9']['FSCA']:.1f} (FSCA) / {R1['sites_to_match_standard9']['Greedy']:.1f} (greedy) sites; "
    f"9 FSCA / greedy sites: {R1['rmse_um']['FSCA'][9]:.3f} / {R1['rmse_um']['Greedy'][9]:.3f} um.",
    f"RQ2   no measurement: map RMSE {R2['rmse_um']['none'][0]:.3f} (mean map) -> {R2['rmse_um']['context'][0]:.3f} (context) -> "
    f"{R2['rmse_um'][BEST][0]:.3f} um ({BEST}, offset model {OFFSET_MODEL}); with 1 / 3 FSCA sites: "
    f"{R2['rmse_um']['none'][1]:.3f} vs {R2['rmse_um'][BEST][1]:.3f} / {R2['rmse_um']['none'][3]:.3f} vs {R2['rmse_um'][BEST][3]:.3f} um; "
    f"measured sites saved by {BEST} at k = 0 / 1 / 3: {R2['sites_saved'][BEST][0]:.2f} / {R2['sites_saved'][BEST][1]:.2f} / "
    f"{R2['sites_saved'][BEST][3]:.2f}.",
    f"FWD   trained on earlier lots only ({int(m_fwd.n_lots.max())} test lots): standard 9 "
    f"{_val(m_fwd, level='none', design='Standard', k=9):.3f}, 9 FSCA {_val(m_fwd, level='none', design='FSCA', k=9):.3f}, "
    f"3 FSCA {_val(m_fwd, level='none', design='FSCA', k=3):.3f} vs {_val(m_fwd, level=BEST, design='FSCA', k=3):.3f} um with {BEST}.",
    f"RQ5   measure-or-skip at tau = {TAUS[-1]} um ({BEST}): {_val(m_skip, 'skipped', level=BEST, tau=TAUS[-1]):.0%} of wafers skipped, "
    f"{_val(m_skip, 'skipped_within_tau', level=BEST, tau=TAUS[-1]):.0%} of them within tolerance (target {1 - ALPHA:.0%}).",
    "PAIRED " + "; ".join(f"{k}: {v['diff_a_minus_b']:+.3f} [{v['ci_lo']:+.3f}, {v['ci_hi']:+.3f}] ({v['lots_a_better']}/{v['n_lots']} lots)"
                         for k, v in P.items()),
]
print("\n".join(lines))
S["headline"] = lines


def _js(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


with open(os.path.join(OUT_DIR, "results.json"), "w") as f:
    json.dump(S, f, indent=1, default=_js)
print("\n===== results.json =====")
print(json.dumps(S, indent=1, default=_js))

import shutil
zip_path = shutil.make_archive(os.path.join(DATA_DIR, "results_sparse_metrology"), "zip", OUT_DIR)
print("\nall results zipped to", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    pass

## 7. Extra analyses reported in the paper (v2.1)
Uses the variables of cells 0-6 and the cached features, so it takes about a minute. It writes `results_extras.json`: the decomposition of the wafer-to-wafer variation, the wafer-mean virtual metrology for each feature set, the measure-or-skip rule forward in time with the learning curve of the prior, and the OES wavelength check (offset of nine F I lines and the effect of correcting the line windows).

In [ ]:
# ---- 7. Extra analyses reported in the paper (run after cells 0-6; uses their variables and the cached features)
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import LeaveOneGroupOut
EX = {}
m_w = Y.mean(1)
order_w = meta.wafer.values.astype(float)

# (a) what the wafer-to-wafer variation is made of: least squares with a lot effect and a common linear drift
#     in the wafer's position (lots 7 and 10 lack their late wafers, so plain lot means would absorb drift)
_L = pd.get_dummies(pd.Series(lots).astype(str)).values.astype(float)
_o = order_w - order_w.mean()
_b = np.linalg.lstsq(np.c_[_L, _o], m_w, rcond=None)[0]
_fl, _fd = _L @ _b[:-1], _o * _b[-1]
tot = ((m_w - m_w.mean()) ** 2).sum()
EX["wafer_mean_variance_shares"] = {"between_lot": float(((_fl - _fl.mean()) ** 2).sum() / tot),
                                    "within_lot_linear_drift": float(((_fd - _fd.mean()) ** 2).sum() / tot),
                                    "rest": float(((m_w - _fl - _fd) ** 2).sum() / tot),
                                    "common_slope_um_per_wafer": float(_b[-1])}
shp = Y - m_w[:, None]
EX["offset_share_of_wafer_to_wafer_variance"] = float(m_w.var() / (m_w.var() + ((shp - shp.mean(0)) ** 2).mean()))
EX["first_wafer_depth_um_by_lot"] = {int(g): float(m_w[(lots == g) & (order_w == order_w[lots == g].min())][0]) for g in np.unique(lots)}


# (b) wafer-mean virtual metrology, leave-one-lot-out, elastic net tuned by inner leave-one-lot-out
def _enet_lolo(X):
    X = np.where(np.isnan(X), np.nanmean(X, 0), X)
    p = np.zeros_like(m_w)
    for g in np.unique(lots):
        tr, te = lots != g, lots == g
        mu_, sd_ = X[tr].mean(0), X[tr].std(0) + 1e-12
        cv = list(LeaveOneGroupOut().split(X[tr], m_w[tr], lots[tr]))
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mdl = ElasticNetCV(l1_ratio=[0.05, 0.3, 0.7, 1.0], n_alphas=25, cv=cv, max_iter=5000, tol=1e-3)
            mdl.fit((X[tr] - mu_) / sd_, m_w[tr])
        p[te] = mdl.predict((X[te] - mu_) / sd_)
    return p


sets = {"context": [C.values], "process": [Fp.values], "context+process": [C.values, Fp.values]}
if Fo is not None:
    sets.update({"OES": [Fo.values], "context+OES": [C.values, Fo.values], "context+process+OES": [C.values, Fp.values, Fo.values]})
vm_rows = {}
for name, parts in sets.items():
    p = _enet_lolo(np.concatenate(parts, 1))
    e = p - m_w
    le = pd.Series(e).groupby(lots).mean()
    vm_rows[name] = {"rmse_um": float(np.sqrt((e ** 2).mean())), "r2": float(1 - (e ** 2).sum() / tot),
                     "macro_lot_mae_um": float(pd.Series(np.abs(e)).groupby(lots).mean().mean()),
                     "mean_error_lots_6_8_um": float(le.loc[[6, 7, 8]].mean()), "mean_error_other_lots_um": float(le.drop([6, 7, 8]).mean())}
e0 = np.array([m_w[lots != g].mean() for g in lots]) - m_w           # training mean of the other lots
vm_rows["training mean"] = {"rmse_um": float(np.sqrt((e0 ** 2).mean())), "r2": float(1 - (e0 ** 2).sum() / tot)}
EX["wafer_mean_vm"] = vm_rows
print("wafer-mean virtual metrology (leave-one-lot-out, elastic net):")
display(pd.DataFrame(vm_rows).T.round(3))

# (c) measure-or-skip forward in time (calibrated on earlier lots only) and the learning curve of the prior
Xb = levels[BEST]
rows_f = []
for j in [g for g in np.unique(lots) if g >= 4]:
    tr, te = lots < j, lots == j
    Ytr, Yte, ltr = Y[tr], Y[te], lots[tr]
    Xtr = None if Xb is None else Xb[tr]
    Xte = None if Xb is None else Xb[te]
    Poof = oof_predict(Ytr, Xtr, ltr, N_MODES)
    res_m = np.abs(Ytr.mean(1) - Poof.mean(1))
    if Xb is None:
        d_oof, d_te = np.ones(len(Ytr)), np.ones(int(te.sum()))
    else:
        la = np.log(res_m + 1e-3)
        d_oof = np.zeros(len(Ytr))
        for g2 in np.unique(ltr):
            m2 = ltr != g2
            d_oof[~m2] = np.exp(ridge_predict(Xtr[m2], la[m2], ltr[m2], Xtr[~m2]))
        d_te = np.exp(ridge_predict(Xtr, la, ltr, Xte))
    q = conformal_quantile(res_m / d_oof, ALPHA)
    mu_f = MapVM(N_MODES).fit(Ytr, Xtr, ltr).predict(Xte, n=int(te.sum()))
    for i in range(len(Yte)):
        err = abs(mu_f[i].mean() - Yte[i].mean())
        for tau in TAUS:
            rows_f.append({"lot": int(j), "n_train_lots": int(len(np.unique(ltr))), "tau": tau, "skipped": bool(q * d_te[i] <= tau),
                           "within": bool(err <= tau), "prior_map_rmse": float(np.sqrt(((mu_f[i] - Yte[i]) ** 2).mean()))})
fs_ = pd.DataFrame(rows_f)
EX["skip_forward"] = {float(t): {"wafers": int(len(g)), "skipped_share": float(g.skipped.mean()),
                                 "skipped_within_tau": (float(g[g.skipped].within.mean()) if g.skipped.any() else None)}
                      for t, g in fs_.groupby("tau")}
if "res_fwd" in globals():                     # learning curve of the prior from the main forward-in-time run (cell 3)
    _lc = res_fwd[(res_fwd.level == BEST) & (res_fwd.design == "FSCA") & (res_fwd.k == 0)]
    EX["prior_learning_curve"] = _lc.groupby("lot").agg(n_train_lots=("n_training_lots", "first"), prior_map_rmse=("rmse", "mean")
                                                        ).reset_index().to_dict(orient="records")
else:
    EX["prior_learning_curve"] = fs_[fs_.tau == TAUS[-1]].groupby("lot").agg(
        n_train_lots=("n_train_lots", "first"), prior_map_rmse=("prior_map_rmse", "mean")).reset_index().to_dict(orient="records")
print("measure-or-skip forward in time:", EX["skip_forward"])

# (d) OES wavelength axis: offset of the strongest F I lines (NIST air wavelengths) in the record-mean spectra
_dbg = os.path.join(OUT_DIR, "oes_debug.npz")
if Fo is not None and os.path.exists(_dbg):
    z_ = np.load(_dbg)
    wl_, ms_ = z_["wavelengths"].astype(float), z_["mean_spec"].astype(float)
    FI = [623.96, 634.85, 677.40, 685.60, 690.25, 703.75, 712.79, 739.87, 775.47]
    offs = []
    for l0 in FI:
        w_ = np.abs(wl_ - l0) <= 3.0
        offs.append(float(np.median([wl_[w_][np.argmax(s_[w_])] - l0 for s_ in ms_])))
    EX["oes_wavelength_offset_nm"] = {"per_line": dict(zip(map(str, FI), offs)), "median": float(np.median(offs)),
                                      "pixel_spacing_nm": float(np.median(np.diff(wl_)))}

    def _lines(shift):
        out = []
        for s_ in ms_:
            b_ = s_[s_ <= np.percentile(s_, 10)].mean()
            t_ = (s_ - b_).sum()
            f_ = {}
            for nm, l0 in OES_LINES_NM.items():
                c_ = l0 + shift
                w_ = np.abs(wl_ - c_) <= 0.8
                sd_ = (np.abs(wl_ - c_) >= 1.5) & (np.abs(wl_ - c_) <= 3.0)
                f_[nm] = (s_[w_].mean() - s_[sd_].mean()) * len(wl_) / t_
            out.append(f_)
        return pd.DataFrame(out, index=list(z_["keys"])).reindex(maps.index)
    L_nom, L_cor = _lines(0.0), _lines(EX["oes_wavelength_offset_nm"]["median"])
    EX["oes_line_windows"] = {"median_rel_intensity_nominal": L_nom.median().round(3).to_dict(),
                              "median_rel_intensity_corrected": L_cor.median().round(3).to_dict()}
    base_X = np.c_[C.values, Fp.values, Fo.values]
    EX["oes_line_windows"]["wafer_mean_rmse_all_features_um"] = float(np.sqrt(((_enet_lolo(base_X) - m_w) ** 2).mean()))
    EX["oes_line_windows"]["wafer_mean_rmse_plus_corrected_lines_um"] = float(np.sqrt(((_enet_lolo(np.c_[base_X, L_cor.values]) - m_w) ** 2).mean()))
    print("OES wavelength offset (nm):", round(EX["oes_wavelength_offset_nm"]["median"], 2),
          "| wafer-mean RMSE with corrected line windows:", round(EX["oes_line_windows"]["wafer_mean_rmse_all_features_um"], 4),
          "->", round(EX["oes_line_windows"]["wafer_mean_rmse_plus_corrected_lines_um"], 4))

with open(os.path.join(OUT_DIR, "results_extras.json"), "w") as f:
    json.dump(EX, f, indent=1, default=_js)
print("written:", os.path.join(OUT_DIR, "results_extras.json"))

## How the outputs map onto the paper
1. **Introduction** - metrology cost; sparse sampling (McLoone et al. 2018; virtual probe); virtual metrology on this dataset: the full 89-site map from traces with Time-LLM (arXiv:2603.23576, no measured sites) and a wafer-mean benchmark (github.com/cpngfan-ui/bosch-etch-virtual-metrology); gap: no lot-held-out study of *where* to measure and *what process data is worth in measured sites*.
2. **Data and audit** - Fig. 1 (mean map, radial profile, shape variability), Fig. 2 (the campaigns differ by a grid symmetry and an offset; agreement after alignment = noise floor), Fig. 3 (within-lot drift).
3. **Method** - Gaussian conditioning with a virtual-metrology prior, residual covariance from out-of-fold residuals, designs (FSCA, QR, greedy), conformal calibration.
4. **Results** - RQ1 Table/Fig. 4a + estimator table + Fig. 5 (the 9-site statistic vs reconstructed CV); RQ2 Table/Fig. 4b (sites saved); forward-in-time table; RQ3 adaptive designs (negative); RQ4 Fig. 6; RQ5 measure-or-skip (report the failure rate of skipped wafers); RQ6 in-line campaign.
5. **Limitations** - one tool, one recipe, 10 lots, test wafers with >99 % open Si; the 89-site maps were measured months later on another profilometer; the edge ring (r ≥ 90 mm) is the dominant non-uniformity.